### Importaciones

In [85]:
from pathlib import Path
from difflib import SequenceMatcher
import re
import unicodedata

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 150)
pd.set_option("display.max_colwidth", 150)

print("Librerías cargadas")

Librerías cargadas


### Definir rutas

In [86]:
PROJECT_ROOT = Path("..").resolve()

LISTINGS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "historical"
    / "idealista_listings_unknown_date.csv"
)

ZONES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "zones_master.csv"
)

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Anuncios:", LISTINGS_PATH)
print("Barrios:", ZONES_PATH)
print("Carpeta intermedia:", INTERIM_DIR)
print("Carpeta procesada:", PROCESSED_DIR)

print("\n¿Existe el archivo de anuncios?", LISTINGS_PATH.exists())
print("¿Existe la tabla de barrios?", ZONES_PATH.exists())

Anuncios: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\raw\historical\idealista_listings_unknown_date.csv
Barrios: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\zones_master.csv
Carpeta intermedia: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim
Carpeta procesada: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed

¿Existe el archivo de anuncios? True
¿Existe la tabla de barrios? True


### Cargar los datos

In [87]:
df_raw = pd.read_csv(LISTINGS_PATH)

zones = pd.read_csv(
    ZONES_PATH,
    dtype={
        "zone_key": "string",
        "neighborhood_code": "string",
        "district_code": "string",
    },
)

df = df_raw.copy()

print("Anuncios cargados:", len(df))
print("Barrios oficiales:", len(zones))

display(df.head())
display(zones.head())

Anuncios cargados: 11826
Barrios oficiales: 131


,provincia,zona,titulo,PrecioActual,PrecioAnterior,metros,habitaciones,ascensor,localizacion,planta,baños,tags,descripcion,Enlace
0,madrid,ciudad-lineal,"Piso en calle de San Marcelo, 22, Ventas, Madrid",355000,0,69,2.0,S,EXTERIOR,5ª,1,"VIVIENDA,LUMINOSO,VISTAS,REFORMADA,PORTERO,EXTERIOR","Particular Vende vivienda totalmente reformada, cocina independiente, salón, dos dormitorios y un baño. Amplias vistas a zonas ajardinadas y porte...",https://www.idealista.com/inmueble/106956987/
1,madrid,carabanchel,"Piso en calle Cabo Nicolás Mur, San Isidro, Madrid",149000,159000,91,3.0,N,EXTERIOR,1ª,0,"PISO,EXCLUSIVA,INMOBILIARIA,OPORTUNIDAD","Inmobiliarias Encuentro vende en exclusiva la NUDA PROPIEDAD PARA INVERSORES Presentamos este fantástico piso en el barrio de San Isidro, Madrid, ...",https://www.idealista.com/inmueble/106906044/
2,madrid,centro,"Piso en calle de Rodas, Lavapiés-Embajadores, Madrid",195000,0,36,1.0,S,NaN,2ª,0,"EXCLUSIVA,ESTUDIO","ESTUDIO EN PLENO CENTRO DE MADRIDSarago Servicios Inmobiliarios presenta, en exclusiva, este fantástico estudio en segunda planta con ascensor, en...",https://www.idealista.com/inmueble/107306175/
3,madrid,usera,"Piso en calle de Ferroviarios, Almendrales, Madrid",195000,0,58,1.0,S,INTERIOR,BAJO,0,"VIVIENDA,HOGAR,FUNCIONAL","Esta acogedora vivienda, ubicada en una planta baja, es perfecta para quienes buscan un hogar listo para entrar a vivir, sin necesidad de realizar...",https://www.idealista.com/inmueble/106325171/
4,madrid,tetuan,"Dúplex en Bellas Vistas, Madrid",715000,750000,140,3.0,S,EXTERIOR,2ª,0,"TERRAZA,EXCLUSIVA,MODERNO","Maravilloso ATICO de reciente construcción, con TERRAZA en el centro de Madrid.En una calle muy tranquila, y RESIDENCIAL, cerca de todos los comer...",https://www.idealista.com/inmueble/106627265/


,zone_key,neighborhood_code,neighborhood_name,district_code,district_name,latitude,longitude,area_km2
0,MAD-011,011,Palacio,01,Centro,40.414971,-3.715596,1.469906
1,MAD-012,012,Embajadores,01,Centro,40.409558,-3.701534,1.033724
2,MAD-013,013,Cortes,01,Centro,40.414445,-3.698547,0.591874
3,MAD-014,014,Justicia,01,Centro,40.423895,-3.695812,0.739413
4,MAD-015,015,Universidad,01,Centro,40.425365,-3.706674,0.948026


### Normalizar textos

In [88]:
def normalizar_texto(valor):
    """
    Convierte un texto en una clave comparable.

    Ejemplo:
    'Lavapiés-Embajadores' -> 'lavapies-embajadores'
    """
    if pd.isna(valor):
        return pd.NA

    texto = str(valor).strip().lower()

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    texto = re.sub(r"[^a-z0-9]+", "-", texto)
    texto = texto.strip("-")

    return texto if texto else pd.NA

### Extraer la subzona indicada en el título

In [89]:
def extraer_subzona_portal(titulo):
    """
    Extrae la zona indicada por el portal al final del título.

    Ejemplos:
    'Piso en calle de Rodas, Lavapiés-Embajadores, Madrid'
        -> 'Lavapiés-Embajadores'

    'Dúplex en Bellas Vistas, Madrid'
        -> 'Bellas Vistas'
    """
    if pd.isna(titulo):
        return pd.NA

    texto = str(titulo).strip()

    # Eliminar la terminación final ", Madrid".
    texto = re.sub(
        r",\s*madrid\s*$",
        "",
        texto,
        flags=re.IGNORECASE,
    )

    partes = [
        parte.strip()
        for parte in texto.split(",")
        if parte.strip()
    ]

    # Cuando hay dirección y zona, la última parte suele ser la zona.
    if len(partes) >= 2:
        return partes[-1]

    # Para títulos como "Piso en Bellas Vistas".
    coincidencia = re.search(
        r"\ben\s+(.+)$",
        texto,
        flags=re.IGNORECASE,
    )

    if coincidencia:
        return coincidencia.group(1).strip()

    return pd.NA

In [90]:
df["subzona_portal"] = df["titulo"].apply(
    extraer_subzona_portal
)

df["subzona_portal_norm"] = df["subzona_portal"].apply(
    normalizar_texto
)

print(
    "Anuncios sin subzona extraída:",
    df["subzona_portal"].isna().sum(),
)

print(
    "Subzonas diferentes:",
    df["subzona_portal_norm"].nunique(),
)

display(
    df[
        [
            "zona",
            "titulo",
            "subzona_portal",
            "subzona_portal_norm",
        ]
    ].head(30)
)

Anuncios sin subzona extraída: 0
Subzonas diferentes: 139


,zona,titulo,subzona_portal,subzona_portal_norm
0,ciudad-lineal,"Piso en calle de San Marcelo, 22, Ventas, Madrid",Ventas,ventas
1,carabanchel,"Piso en calle Cabo Nicolás Mur, San Isidro, Madrid",San Isidro,san-isidro
2,centro,"Piso en calle de Rodas, Lavapiés-Embajadores, Madrid",Lavapiés-Embajadores,lavapies-embajadores
3,usera,"Piso en calle de Ferroviarios, Almendrales, Madrid",Almendrales,almendrales
4,tetuan,"Dúplex en Bellas Vistas, Madrid",Bellas Vistas,bellas-vistas
5,arganzuela,"Piso en Mazarredo, 18, Imperial, Madrid",Imperial,imperial
6,barrio-de-salamanca,"Piso en calle de Núñez de Balboa, Recoletos, Madrid",Recoletos,recoletos
7,puente-de-vallecas,"Piso en calle de Rodríguez Espinosa, s/n, Palomeras Bajas, Madrid",Palomeras Bajas,palomeras-bajas
8,moncloa,"Chalet adosado en calle de la Isla de Alegranza, 51, Ciudad Universitaria, Madrid",Ciudad Universitaria,ciudad-universitaria
9,chamberi,"Piso en Blanca de Navarra, 4, Almagro, Madrid",Almagro,almagro


### Normalizar los distritos actuales

In [91]:
district_name_map = {
    "arganzuela": "Arganzuela",
    "barajas": "Barajas",
    "barrio-de-salamanca": "Salamanca",
    "carabanchel": "Carabanchel",
    "centro": "Centro",
    "chamartin": "Chamartín",
    "chamberi": "Chamberí",
    "ciudad-lineal": "Ciudad Lineal",
    "fuencarral": "Fuencarral-El Pardo",
    "hortaleza": "Hortaleza",
    "latina": "Latina",
    "moncloa": "Moncloa-Aravaca",
    "moratalaz": "Moratalaz",
    "puente-de-vallecas": "Puente de Vallecas",
    "retiro": "Retiro",
    "san-blas": "San Blas-Canillejas",
    "tetuan": "Tetuán",
    "usera": "Usera",
    "vicalvaro": "Vicálvaro",
    "villa-de-vallecas": "Villa de Vallecas",
    "villaverde": "Villaverde",
}

df["district_name_official"] = df["zona"].map(
    district_name_map
)

df["district_name_norm"] = df[
    "district_name_official"
].apply(normalizar_texto)

print(
    "Anuncios sin distrito oficial asignado:",
    df["district_name_official"].isna().sum(),
)

display(
    df[
        [
            "zona",
            "district_name_official",
        ]
    ].drop_duplicates().sort_values("zona")
)

Anuncios sin distrito oficial asignado: 0


,zona,district_name_official
5,arganzuela,Arganzuela
73,barajas,Barajas
6,barrio-de-salamanca,Salamanca
1,carabanchel,Carabanchel
2,centro,Centro
11,chamartin,Chamartín
9,chamberi,Chamberí
0,ciudad-lineal,Ciudad Lineal
18,fuencarral,Fuencarral-El Pardo
36,hortaleza,Hortaleza


### Preparar los nombres oficiales de barrios

In [92]:
zones["neighborhood_name_norm"] = zones[
    "neighborhood_name"
].apply(normalizar_texto)

zones["district_name_norm"] = zones[
    "district_name"
].apply(normalizar_texto)

display(
    zones[
        [
            "zone_key",
            "neighborhood_name",
            "neighborhood_name_norm",
            "district_name",
        ]
    ].head(20)
)

,zone_key,neighborhood_name,neighborhood_name_norm,district_name
0,MAD-011,Palacio,palacio,Centro
1,MAD-012,Embajadores,embajadores,Centro
2,MAD-013,Cortes,cortes,Centro
3,MAD-014,Justicia,justicia,Centro
4,MAD-015,Universidad,universidad,Centro
5,MAD-016,Sol,sol,Centro
6,MAD-021,Imperial,imperial,Arganzuela
7,MAD-022,Acacias,acacias,Arganzuela
8,MAD-023,Chopera,chopera,Arganzuela
9,MAD-024,Legazpi,legazpi,Arganzuela


### Intentar una correspondencia exacta

In [93]:
neighborhood_lookup = zones[
    [
        "zone_key",
        "neighborhood_code",
        "neighborhood_name",
        "district_code",
        "district_name",
        "neighborhood_name_norm",
        "district_name_norm",
    ]
].copy()

df_matched = df.merge(
    neighborhood_lookup,
    how="left",
    left_on=[
        "subzona_portal_norm",
        "district_name_norm",
    ],
    right_on=[
        "neighborhood_name_norm",
        "district_name_norm",
    ],
    validate="many_to_one",
)

### Calcular la cobertura conseguida

In [94]:
total_anuncios = len(df_matched)

anuncios_asignados = (
    df_matched["zone_key"].notna().sum()
)

anuncios_no_asignados = (
    df_matched["zone_key"].isna().sum()
)

porcentaje_asignado = (
    anuncios_asignados / total_anuncios * 100
)

print("Total de anuncios:", total_anuncios)
print("Anuncios asignados exactamente:", anuncios_asignados)
print("Anuncios no asignados:", anuncios_no_asignados)
print(
    "Porcentaje de asignación exacta:",
    round(porcentaje_asignado, 2),
    "%",
)

Total de anuncios: 11826
Anuncios asignados exactamente: 9002
Anuncios no asignados: 2824
Porcentaje de asignación exacta: 76.12 %


### Revisar las subzonas no asignadas

In [95]:
unmatched_subzones = (
    df_matched.loc[
        df_matched["zone_key"].isna(),
        [
            "zona",
            "district_name_official",
            "subzona_portal",
            "subzona_portal_norm",
        ],
    ]
    .groupby(
        [
            "zona",
            "district_name_official",
            "subzona_portal",
            "subzona_portal_norm",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(columns={"size": "numero_anuncios"})
    .sort_values(
        "numero_anuncios",
        ascending=False,
    )
)

print(
    "Subzonas diferentes sin correspondencia:",
    len(unmatched_subzones),
)

display(unmatched_subzones.head(100))

Subzonas diferentes sin correspondencia: 32


,zona,district_name_official,subzona_portal,subzona_portal_norm,numero_anuncios
6,centro,Centro,Malasaña-Universidad,malasana-universidad,451
5,centro,Centro,Lavapiés-Embajadores,lavapies-embajadores,366
3,centro,Centro,Chueca-Justicia,chueca-justicia,277
4,centro,Centro,Huertas-Cortes,huertas-cortes,177
31,villaverde,Villaverde,Villaverde Alto,villaverde-alto,152
7,chamartin,Chamartín,Bernabéu-Hispanoamérica,bernabeu-hispanoamerica,139
8,chamberi,Chamberí,Nuevos Ministerios-Ríos Rosas,nuevos-ministerios-rios-rosas,125
14,hortaleza,Hortaleza,Conde Orgaz-Piovera,conde-orgaz-piovera,105
28,villa-de-vallecas,Villa de Vallecas,Ensanche de Vallecas - La Gavia,ensanche-de-vallecas-la-gavia,102
18,retiro,Retiro,Jerónimos,jeronimos,96


In [96]:
unmatched_path = (
    INTERIM_DIR
    / "unmatched_portal_subzones.csv"
)

unmatched_subzones.to_csv(
    unmatched_path,
    index=False,
    encoding="utf-8-sig",
)

print("Tabla guardada en:")
print(unmatched_path)

Tabla guardada en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\unmatched_portal_subzones.csv


### Revisar los anuncios correctamente asignados

In [97]:
matched_examples = df_matched.loc[
    df_matched["zone_key"].notna(),
    [
        "titulo",
        "zona",
        "subzona_portal",
        "neighborhood_name",
        "district_name",
        "zone_key",
    ],
].sample(
    n=min(30, anuncios_asignados),
    random_state=42,
)

display(matched_examples)

,titulo,zona,subzona_portal,neighborhood_name,district_name,zone_key
4527,"Piso en calle de Ayala, Goya, Madrid",barrio-de-salamanca,Goya,Goya,Salamanca,MAD-042
3269,"Piso en Los Rosales, Madrid",villaverde,Los Rosales,Los Rosales,Villaverde,MAD-174
767,"Piso en paseo de Eduardo Dato, Almagro, Madrid",chamberi,Almagro,Almagro,Chamberí,MAD-074
7064,"Piso en Palacio, Madrid",centro,Palacio,Palacio,Centro,MAD-011
11374,"Piso en Arapiles, Madrid",chamberi,Arapiles,Arapiles,Chamberí,MAD-072
9019,"Piso en Sol, Madrid",centro,Sol,Sol,Centro,MAD-016
4776,"Piso en paseo de los Olivos, Puerta del Ángel, Madrid",latina,Puerta del Ángel,Puerta del Ángel,Latina,MAD-102
10139,"Estudio en calle de los Mancebos, Palacio, Madrid",centro,Palacio,Palacio,Centro,MAD-011
9005,"Piso en calle de Esteban Carros, 59, Entrevías, Madrid",puente-de-vallecas,Entrevías,Entrevías,Puente de Vallecas,MAD-131
9751,"Piso en Palacio, Madrid",centro,Palacio,Palacio,Centro,MAD-011


In [98]:
# Comparamos los nombres después de normalizarlos,
# evitando falsos errores por tildes, espacios o guiones.

df_matched["district_listing_check"] = (
    df_matched["district_name_official"]
    .apply(normalizar_texto)
)

df_matched["district_zones_check"] = (
    df_matched["district_name"]
    .apply(normalizar_texto)
)

errores_distrito = df_matched.loc[
    df_matched["zone_key"].notna()
    & (
        df_matched["district_listing_check"]
        != df_matched["district_zones_check"]
    ),
    [
        "titulo",
        "zona",
        "district_name_official",
        "district_name",
        "subzona_portal",
        "neighborhood_name",
        "zone_key",
    ],
]

print(
    "Asignaciones realmente inconsistentes:",
    len(errores_distrito),
)

display(errores_distrito.head(20))

Asignaciones realmente inconsistentes: 0


,titulo,zona,district_name_official,district_name,subzona_portal,neighborhood_name,zone_key


### Guardar los resultados provisionales

In [99]:
matched_path = (
    INTERIM_DIR
    / "listings_with_exact_neighborhood_match.csv"
)

unmatched_path = (
    INTERIM_DIR
    / "unmatched_portal_subzones.csv"
)

df_matched.to_csv(
    matched_path,
    index=False,
    encoding="utf-8-sig",
)

unmatched_subzones.to_csv(
    unmatched_path,
    index=False,
    encoding="utf-8-sig",
)

print("Archivo provisional guardado en:")
print(matched_path)

print("\nSubzonas pendientes guardadas en:")
print(unmatched_path)

Archivo provisional guardado en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\listings_with_exact_neighborhood_match.csv

Subzonas pendientes guardadas en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\unmatched_portal_subzones.csv


### resumen de subzonas pendientes

In [100]:
unmatched_subzones = (
    df_matched.loc[
        df_matched["zone_key"].isna(),
        [
            "zona",
            "district_name_official",
            "subzona_portal",
            "subzona_portal_norm",
        ],
    ]
    .groupby(
        [
            "zona",
            "district_name_official",
            "subzona_portal",
            "subzona_portal_norm",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(columns={"size": "numero_anuncios"})
    .sort_values("numero_anuncios", ascending=False)
    .reset_index(drop=True)
)

print(
    "Subzonas diferentes sin correspondencia:",
    len(unmatched_subzones)
)

print(
    "Anuncios representados:",
    unmatched_subzones["numero_anuncios"].sum()
)

display(unmatched_subzones.head(100))

Subzonas diferentes sin correspondencia: 32
Anuncios representados: 2824


,zona,district_name_official,subzona_portal,subzona_portal_norm,numero_anuncios
0,centro,Centro,Malasaña-Universidad,malasana-universidad,451
1,centro,Centro,Lavapiés-Embajadores,lavapies-embajadores,366
2,centro,Centro,Chueca-Justicia,chueca-justicia,277
3,centro,Centro,Huertas-Cortes,huertas-cortes,177
4,villaverde,Villaverde,Villaverde Alto,villaverde-alto,152
5,chamartin,Chamartín,Bernabéu-Hispanoamérica,bernabeu-hispanoamerica,139
6,chamberi,Chamberí,Nuevos Ministerios-Ríos Rosas,nuevos-ministerios-rios-rosas,125
7,hortaleza,Hortaleza,Conde Orgaz-Piovera,conde-orgaz-piovera,105
8,villa-de-vallecas,Villa de Vallecas,Ensanche de Vallecas - La Gavia,ensanche-de-vallecas-la-gavia,102
9,retiro,Retiro,Jerónimos,jeronimos,96


### Generar los tres barrios oficiales candidatos

In [101]:
def calcular_similitud(texto_1, texto_2):
    """
    Devuelve una similitud entre 0 y 1.
    Un valor próximo a 1 indica que los textos son muy similares.
    """
    if pd.isna(texto_1) or pd.isna(texto_2):
        return 0.0

    return SequenceMatcher(
        None,
        str(texto_1),
        str(texto_2),
    ).ratio()


def buscar_candidatos_barrio(fila):
    subzona_norm = fila["subzona_portal_norm"]

    distrito_norm = normalizar_texto(
        fila["district_name_official"]
    )

    candidatos = zones.loc[
        zones["district_name_norm"] == distrito_norm,
        [
            "zone_key",
            "neighborhood_name",
            "neighborhood_name_norm",
        ],
    ].copy()

    # Comprobamos si el nombre oficial aparece dentro
    # de la denominación utilizada por el portal.
    candidatos["coincidencia_contenida"] = candidatos[
        "neighborhood_name_norm"
    ].apply(
        lambda barrio: (
            barrio in subzona_norm
            or subzona_norm in barrio
        )
        if pd.notna(barrio) and pd.notna(subzona_norm)
        else False
    )

    candidatos["similarity_score"] = candidatos[
        "neighborhood_name_norm"
    ].apply(
        lambda barrio: calcular_similitud(
            subzona_norm,
            barrio,
        )
    )

    candidatos = candidatos.sort_values(
        [
            "coincidencia_contenida",
            "similarity_score",
        ],
        ascending=[False, False],
    ).reset_index(drop=True)

    resultado = {
        "numero_coincidencias_contenidas": int(
            candidatos["coincidencia_contenida"].sum()
        )
    }

    for posicion in range(3):
        numero = posicion + 1

        if posicion < len(candidatos):
            candidato = candidatos.iloc[posicion]

            resultado[f"candidate_{numero}_name"] = (
                candidato["neighborhood_name"]
            )

            resultado[f"candidate_{numero}_zone_key"] = (
                candidato["zone_key"]
            )

            resultado[f"candidate_{numero}_score"] = round(
                candidato["similarity_score"],
                3,
            )

            resultado[f"candidate_{numero}_contained"] = bool(
                candidato["coincidencia_contenida"]
            )
        else:
            resultado[f"candidate_{numero}_name"] = pd.NA
            resultado[f"candidate_{numero}_zone_key"] = pd.NA
            resultado[f"candidate_{numero}_score"] = pd.NA
            resultado[f"candidate_{numero}_contained"] = False

    return pd.Series(resultado)

### Aplicar la búsqueda a las 32 subzonas

In [102]:
candidate_results = unmatched_subzones.apply(
    buscar_candidatos_barrio,
    axis=1,
)

mapping_review = pd.concat(
    [
        unmatched_subzones.reset_index(drop=True),
        candidate_results.reset_index(drop=True),
    ],
    axis=1,
)

# Columnas que utilizaremos para registrar
# la decisión definitiva.
mapping_review["selected_zone_key"] = pd.NA
mapping_review["mapping_status"] = "pending_review"
mapping_review["mapping_notes"] = pd.NA

print("Subzonas pendientes:", len(mapping_review))

display(
    mapping_review[
        [
            "district_name_official",
            "subzona_portal",
            "numero_anuncios",
            "candidate_1_name",
            "candidate_1_zone_key",
            "candidate_1_score",
            "candidate_1_contained",
            "candidate_2_name",
            "candidate_2_score",
            "candidate_3_name",
            "candidate_3_score",
        ]
    ]
)

Subzonas pendientes: 32


,district_name_official,subzona_portal,numero_anuncios,candidate_1_name,candidate_1_zone_key,candidate_1_score,candidate_1_contained,candidate_2_name,candidate_2_score,candidate_3_name,candidate_3_score
0,Centro,Malasaña-Universidad,451,Universidad,MAD-015,0.710,True,Palacio,0.296,Embajadores,0.258
1,Centro,Lavapiés-Embajadores,366,Embajadores,MAD-012,0.710,True,Palacio,0.296,Universidad,0.258
2,Centro,Chueca-Justicia,277,Justicia,MAD-014,0.696,True,Embajadores,0.308,Cortes,0.286
3,Centro,Huertas-Cortes,177,Cortes,MAD-013,0.600,True,Embajadores,0.480,Universidad,0.320
4,Villaverde,Villaverde Alto,152,Villaverde Alto - Casco Histórico de Villaverde,MAD-171,0.500,True,Los Rosales,0.308,San Cristóbal,0.214
5,Chamartín,Bernabéu-Hispanoamérica,139,Hispanoamérica,MAD-054,0.757,True,Nueva España,0.457,Prosperidad,0.412
6,Chamberí,Nuevos Ministerios-Ríos Rosas,125,Ríos Rosas,MAD-075,0.513,True,Vallehermoso,0.195,Almagro,0.167
7,Hortaleza,Conde Orgaz-Piovera,105,Piovera,MAD-162,0.538,True,Pinar del Rey,0.375,Canillas,0.222
8,Villa de Vallecas,Ensanche de Vallecas - La Gavia,102,Ensanche de Vallecas,MAD-183,0.816,True,Casco Histórico de Vallecas,0.536,Santa Eugenia,0.381
9,Retiro,Jerónimos,96,Los Jerónimos,MAD-035,0.818,True,Niño Jesús,0.316,Adelfas,0.250


### Guardar la tabla de revisión

In [103]:
mapping_review_path = (
    INTERIM_DIR
    / "portal_subzone_mapping_review.csv"
)

mapping_review.to_csv(
    mapping_review_path,
    index=False,
    encoding="utf-8-sig",
)

print("Tabla de revisión guardada en:")
print(mapping_review_path)

Tabla de revisión guardada en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\portal_subzone_mapping_review.csv


In [104]:
# Creamos una tabla de validación trazable a partir
# de las propuestas de correspondencia.

mapping_validation = mapping_review.copy()

mapping_validation["selected_official_neighborhood"] = pd.NA
mapping_validation["validation_method"] = pd.NA
mapping_validation["confidence"] = pd.NA
mapping_validation["source_name"] = pd.NA
mapping_validation["source_reference"] = pd.NA
mapping_validation["validation_notes"] = pd.NA

# Regla de alta confianza:
# la denominación del portal contiene exactamente
# un barrio oficial y ambos pertenecen al mismo distrito.
high_confidence_mask = (
    mapping_validation["candidate_1_contained"].eq(True)
    & mapping_validation[
        "numero_coincidencias_contenidas"
    ].eq(1)
)

mapping_validation.loc[
    high_confidence_mask,
    "selected_official_neighborhood",
] = mapping_validation.loc[
    high_confidence_mask,
    "candidate_1_name",
]

mapping_validation.loc[
    high_confidence_mask,
    "selected_zone_key",
] = mapping_validation.loc[
    high_confidence_mask,
    "candidate_1_zone_key",
]

mapping_validation.loc[
    high_confidence_mask,
    "mapping_status",
] = "proposed_high_confidence"

mapping_validation.loc[
    high_confidence_mask,
    "validation_method",
] = "official_name_contained_same_district"

mapping_validation.loc[
    high_confidence_mask,
    "confidence",
] = "high"

mapping_validation.loc[
    high_confidence_mask,
    "source_name",
] = "División oficial de barrios de Madrid"

mapping_validation.loc[
    high_confidence_mask,
    "source_reference",
] = "data/processed/zones_master.csv"

mapping_validation.loc[
    high_confidence_mask,
    "validation_notes",
] = (
    "La denominación del portal contiene exactamente "
    "un nombre de barrio oficial perteneciente al mismo distrito."
)

print(
    "Propuestas de alta confianza:",
    high_confidence_mask.sum(),
)

print(
    "Anuncios cubiertos por estas propuestas:",
    mapping_validation.loc[
        high_confidence_mask,
        "numero_anuncios",
    ].sum(),
)

Propuestas de alta confianza: 19
Anuncios cubiertos por estas propuestas: 2439


### casos que necesitan revisión documental

In [105]:
pending_validation = mapping_validation.loc[
    ~high_confidence_mask,
    [
        "district_name_official",
        "subzona_portal",
        "numero_anuncios",
        "candidate_1_name",
        "candidate_1_zone_key",
        "candidate_2_name",
        "candidate_2_zone_key",
        "numero_coincidencias_contenidas",
    ],
].copy()

print(
    "Subzonas que requieren revisión individual:",
    len(pending_validation),
)

print(
    "Anuncios afectados:",
    pending_validation["numero_anuncios"].sum(),
)

display(pending_validation)

Subzonas que requieren revisión individual: 13
Anuncios afectados: 385


,district_name_official,subzona_portal,numero_anuncios,candidate_1_name,candidate_1_zone_key,candidate_2_name,candidate_2_zone_key,numero_coincidencias_contenidas
12,Carabanchel,Buena Vista,77,Buenavista,MAD-116,Puerta Bonita,MAD-115,0
14,Hortaleza,Sanchinarro,55,Apóstol Santiago,MAD-165,Canillas,MAD-163,0
16,Vicálvaro,Los Berrocales,46,Valderrivas,MAD-193,El Cañaveral,MAD-194,0
18,Fuencarral-El Pardo,Las Tablas,43,La Paz,MAD-085,Mirasierra,MAD-087,0
21,Carabanchel,Pau de Carabanchel,33,Opañel,MAD-112,Abrantes,MAD-117,0
22,Hortaleza,Virgen del Cortijo - Manoteras,30,Pinar del Rey,MAD-164,Valdefuentes,MAD-166,0
24,Vicálvaro,Los Ahijones,24,El Cañaveral,MAD-194,Casco Histórico de Vicálvaro,MAD-191,0
25,Vicálvaro,Los Cerros,20,Valderrivas,MAD-193,El Cañaveral,MAD-194,0
26,Vicálvaro,Ambroz,18,Valdebernardo,MAD-192,Valderrivas,MAD-193,0
28,Fuencarral-El Pardo,Montecarmelo,12,Fuentelarreina,MAD-082,El Pardo,MAD-081,0


### Guardar la tabla de trabajo

In [106]:
validation_path = (
    INTERIM_DIR
    / "portal_subzone_mapping_validation.csv"
)

mapping_validation.to_csv(
    validation_path,
    index=False,
    encoding="utf-8-sig",
)

print("Tabla de validación guardada en:")
print(validation_path)

Tabla de validación guardada en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\portal_subzone_mapping_validation.csv


### Validación documental de las 13 subzonas pendientes

In [107]:
mapping_validated = mapping_validation.copy()


# ============================================================
# Validación documental de las 13 correspondencias pendientes
# ============================================================

manual_cases = [
    {
        "subzona_portal": "Buena Vista",
        "selected_official_neighborhood": "Buenavista",
        "selected_zone_key": "MAD-116",
        "mapping_status": "validated",
        "validation_method": "orthographic_variant",
        "confidence": "high",
        "source_name": "División oficial de barrios de Madrid",
        "source_reference": "data/processed/zones_master.csv",
        "validation_notes": (
            "Variante ortográfica del nombre oficial Buenavista. "
            "Solo se elimina el espacio entre las dos palabras."
        ),
    },
    {
        "subzona_portal": "Sanchinarro",
        "selected_official_neighborhood": "Valdefuentes",
        "selected_zone_key": "MAD-166",
        "mapping_status": "validated",
        "validation_method": "official_municipal_location",
        "confidence": "high",
        "source_name": (
            "Ayuntamiento de Madrid - Sala de estudio Sanchinarro"
        ),
        "source_reference": (
            "https://www.madrid.es/portales/munimadrid/es/Inicio/"
            "Educacion-y-empleo/Educacion/Direcciones-y-telefonos/"
            "Sala-de-estudio-Sanchinarro-Hortaleza-/"
            "?vgnextoid=459b10e528908310VgnVCM1000000b205a0aRCRD"
        ),
        "validation_notes": (
            "La ficha municipal de una instalación situada en "
            "Sanchinarro identifica su barrio oficial como Valdefuentes."
        ),
    },
    {
        "subzona_portal": "Los Berrocales",
        "selected_official_neighborhood": (
            "Casco Histórico de Vicálvaro"
        ),
        "selected_zone_key": "MAD-191",
        "mapping_status": "validated",
        "validation_method": "official_urban_development",
        "confidence": "high",
        "source_name": "Madrid Crece - Los Berrocales",
        "source_reference": (
            "https://madridcrece.madrid.es/"
            "nuevos-barrios/los-berrocales/"
        ),
        "validation_notes": (
            "La ficha municipal del desarrollo sitúa Los Berrocales "
            "en Vicálvaro - Casco Histórico de Vicálvaro."
        ),
    },
    {
        "subzona_portal": "Las Tablas",
        "selected_official_neighborhood": "Valverde",
        "selected_zone_key": "MAD-086",
        "mapping_status": "validated",
        "validation_method": "official_municipal_location",
        "confidence": "high",
        "source_name": (
            "Ayuntamiento de Madrid - Espacio para mayores Las Tablas"
        ),
        "source_reference": (
            "https://www.madrid.es/portales/munimadrid/es/Inicio/"
            "Servicios-sociales-y-salud/Servicios-sociales/"
            "Direcciones-y-telefonos/Espacio-para-mayores-Las-Tablas/"
            "?vgnextoid=2c22864474449810VgnVCM1000001d4a900aRCRD"
        ),
        "validation_notes": (
            "La ficha municipal de una instalación de Las Tablas "
            "identifica su barrio oficial como Valverde."
        ),
    },
    {
        "subzona_portal": "Pau de Carabanchel",
        "selected_official_neighborhood": "Buenavista",
        "selected_zone_key": "MAD-116",
        "mapping_status": "validated",
        "validation_method": "official_municipal_location",
        "confidence": "high",
        "source_name": (
            "Ayuntamiento de Madrid - "
            "Equipamientos de Carabanchel Alto"
        ),
        "source_reference": (
            "https://www.madrid.es/portales/munimadrid/es/Inicio/"
            "Infancia-y-familia/Direcciones-y-telefonos/"
            "Colegio-Publico-de-Educacion-Especial-Maria-Soriano/"
            "?vgnextoid=11dc3b54d771c010VgnVCM1000000b205a0aRCRD"
        ),
        "validation_notes": (
            "Los equipamientos municipales situados en el entorno "
            "del PAU y de la avenida de la Peseta figuran en el "
            "barrio oficial de Buenavista."
        ),
    },
    {
        "subzona_portal": "Virgen del Cortijo - Manoteras",
        "selected_official_neighborhood": "Valdefuentes",
        "selected_zone_key": "MAD-166",
        "mapping_status": "validated",
        "validation_method": "official_municipal_location",
        "confidence": "high",
        "source_name": (
            "Ayuntamiento de Madrid - "
            "Centro de Salud Virgen del Cortijo"
        ),
        "source_reference": (
            "https://www.madrid.es/portales/munimadrid/es/Inicio/"
            "El-Ayuntamiento/Hortaleza/"
            "Centro-de-Salud-Virgen-del-Cortijo/"
            "?vgnextoid=bc7e4b6faa61c010VgnVCM1000000b205a0aRCRD"
        ),
        "validation_notes": (
            "Las fichas municipales de Virgen del Cortijo y "
            "Manoteras identifican el barrio oficial como Valdefuentes."
        ),
    },
    {
        "subzona_portal": "Los Ahijones",
        "selected_official_neighborhood": (
            "Casco Histórico de Vicálvaro"
        ),
        "selected_zone_key": "MAD-191",
        "mapping_status": "validated",
        "validation_method": "official_urban_development",
        "confidence": "high",
        "source_name": "Madrid Crece - Los Ahijones",
        "source_reference": (
            "https://madridcrece.madrid.es/"
            "nuevos-barrios/los-ahijones/"
        ),
        "validation_notes": (
            "La ficha municipal del desarrollo sitúa Los Ahijones "
            "en Vicálvaro - Casco Histórico de Vicálvaro."
        ),
    },
    {
        "subzona_portal": "Los Cerros",
        "selected_official_neighborhood": "El Cañaveral",
        "selected_zone_key": "MAD-194",
        "mapping_status": "validated",
        "validation_method": "official_urban_development",
        "confidence": "high",
        "source_name": "Madrid Crece - Los Cerros",
        "source_reference": (
            "https://madridcrece.madrid.es/"
            "nuevos-barrios/los-cerros/"
        ),
        "validation_notes": (
            "La ficha municipal del desarrollo sitúa Los Cerros "
            "en Vicálvaro - El Cañaveral."
        ),
    },
    {
        "subzona_portal": "Ambroz",
        "selected_official_neighborhood": (
            "Casco Histórico de Vicálvaro"
        ),
        "selected_zone_key": "MAD-191",
        "mapping_status": "validated",
        "validation_method": "official_statistical_correspondence",
        "confidence": "high",
        "source_name": (
            "Banco de Datos del Ayuntamiento de Madrid - "
            "Correspondencia territorial de Idealista"
        ),
        "source_reference": (
            "https://servpub.madrid.es/CSEBD_WBINTER/"
            "seleccionSerie.html?numSerie=0504030000153"
        ),
        "validation_notes": (
            "La correspondencia territorial publicada por el "
            "Ayuntamiento asigna la subzona Ambroz al barrio oficial "
            "Casco Histórico de Vicálvaro."
        ),
    },
    {
        "subzona_portal": "Montecarmelo",
        "selected_official_neighborhood": "El Goloso",
        "selected_zone_key": "MAD-088",
        "mapping_status": "validated",
        "validation_method": "official_municipal_location",
        "confidence": "high",
        "source_name": (
            "Ayuntamiento de Madrid - Sala de estudio Montecarmelo"
        ),
        "source_reference": (
            "https://www.madrid.es/portales/munimadrid/es/Inicio/"
            "Destacados-Home/"
            "Sala-de-estudio-Montecarmelo-Fuencarral-El-Pardo-/"
            "?vgnextoid=ae87964bd47a9210VgnVCM1000000b205a0aRCRD"
        ),
        "validation_notes": (
            "La ficha municipal de Montecarmelo identifica el "
            "barrio oficial como El Goloso."
        ),
    },
    {
        "subzona_portal": "Arroyo del Fresno",
        "selected_official_neighborhood": "Mirasierra",
        "selected_zone_key": "MAD-087",
        "mapping_status": "validated",
        "validation_method": "official_municipal_location",
        "confidence": "high",
        "source_name": (
            "Ayuntamiento de Madrid - Parque Arroyo Fresno"
        ),
        "source_reference": (
            "https://www.madrid.es/portales/munimadrid/es/"
            "Parque-Arroyo-Fresno-Tomas-y-Valiente/"
            "?vgnextoid=915fc8c5494be210VgnVCM2000000c205a0aRCRD"
        ),
        "validation_notes": (
            "La ficha municipal del parque Arroyo Fresno "
            "identifica el barrio oficial como Mirasierra."
        ),
    },
    {
        "subzona_portal": "Valdebernardo - Valderrivas",
        "selected_official_neighborhood": "Valderrivas",
        "selected_zone_key": "MAD-193",
        "mapping_status": "validated",
        "validation_method": "official_statistical_correspondence",
        "confidence": "high",
        "source_name": (
            "Banco de Datos del Ayuntamiento de Madrid - "
            "Correspondencia territorial de Idealista"
        ),
        "source_reference": (
            "https://servpub.madrid.es/CSEBD_WBINTER/"
            "seleccionSerie.html?numSerie=0504030000153"
        ),
        "validation_notes": (
            "La correspondencia territorial publicada por el "
            "Ayuntamiento asigna la denominación agregada "
            "Valdebernardo-Valderrivas al barrio oficial Valderrivas."
        ),
    },
    {
        "subzona_portal": "Valdecarros",
        "selected_official_neighborhood": (
            "Casco Histórico de Vallecas"
        ),
        "selected_zone_key": "MAD-181",
        "mapping_status": "validated",
        "validation_method": "official_urban_development",
        "confidence": "high",
        "source_name": "Madrid Crece - Valdecarros",
        "source_reference": (
            "https://madridcrece.madrid.es/"
            "nuevos-barrios/valdecarros/"
        ),
        "validation_notes": (
            "La ficha municipal del desarrollo sitúa Valdecarros "
            "en Villa de Vallecas - Casco Histórico de Vallecas."
        ),
    },
]


# ============================================================
# Comprobaciones previas
# ============================================================

# Deben existir exactamente 13 decisiones manuales.

if len(manual_cases) != 13:
    raise ValueError(
        "Se esperaban 13 correspondencias manuales, "
        f"pero se han definido {len(manual_cases)}."
    )


# No debe aparecer ninguna subzona duplicada.

manual_subzones = [
    case["subzona_portal"]
    for case in manual_cases
]

duplicated_manual_subzones = (
    pd.Series(manual_subzones)
    .loc[
        pd.Series(manual_subzones).duplicated(
            keep=False
        )
    ]
    .unique()
    .tolist()
)

if duplicated_manual_subzones:
    raise ValueError(
        "Hay subzonas duplicadas en manual_cases: "
        f"{duplicated_manual_subzones}"
    )


# Todas las decisiones deben contener los mismos campos.

validation_columns = [
    "selected_official_neighborhood",
    "selected_zone_key",
    "mapping_status",
    "validation_method",
    "confidence",
    "source_name",
    "source_reference",
    "validation_notes",
]

required_case_columns = [
    "subzona_portal",
    *validation_columns,
]

for case in manual_cases:
    missing_columns = [
        column
        for column in required_case_columns
        if column not in case
    ]

    if missing_columns:
        raise ValueError(
            f"El caso {case.get('subzona_portal', 'desconocido')} "
            f"no contiene las columnas: {missing_columns}"
        )


# ============================================================
# Aplicación de las decisiones documentales
# ============================================================

for case in manual_cases:
    mask = (
        mapping_validated["subzona_portal"]
        == case["subzona_portal"]
    )

    if mask.sum() != 1:
        raise ValueError(
            f"Se esperaba exactamente una fila para "
            f"{case['subzona_portal']}, "
            f"pero se encontraron {mask.sum()}."
        )

    for column in validation_columns:
        mapping_validated.loc[
            mask,
            column,
        ] = case[column]


# ============================================================
# Validación final
# ============================================================

pending_rows = (
    mapping_validated["mapping_status"]
    == "pending_review"
).sum()

validated_manual_rows = (
    mapping_validated["subzona_portal"]
    .isin(manual_subzones)
    & (
        mapping_validated["mapping_status"]
        == "validated"
    )
).sum()


print(
    "Casos documentales incorporados:",
    len(manual_cases),
)

print(
    "Casos manuales validados:",
    validated_manual_rows,
)

print(
    "Filas todavía pendientes:",
    pending_rows,
)


if validated_manual_rows != len(manual_cases):
    raise AssertionError(
        "No se han validado correctamente todos los casos manuales."
    )

if pending_rows != 0:
    raise AssertionError(
        "Todavía quedan correspondencias pendientes de revisión."
    )

Casos documentales incorporados: 13
Casos manuales validados: 13
Filas todavía pendientes: 0


### Comprueba los códigos contra zones_master

In [108]:
valid_zone_keys = set(
    zones["zone_key"].dropna()
)

invalid_zone_keys = (
    mapping_validated.loc[
        mapping_validated["selected_zone_key"].notna()
        & ~mapping_validated["selected_zone_key"].isin(
            valid_zone_keys
        ),
        [
            "subzona_portal",
            "selected_official_neighborhood",
            "selected_zone_key",
        ],
    ]
)

print(
    "Códigos de barrio inexistentes:",
    len(invalid_zone_keys),
)

display(invalid_zone_keys)

Códigos de barrio inexistentes: 0


,subzona_portal,selected_official_neighborhood,selected_zone_key


### Guarda la tabla definitiva

In [109]:
validated_path = (
    INTERIM_DIR
    / "portal_subzone_mapping_validated.csv"
)

mapping_validated.to_csv(
    validated_path,
    index=False,
    encoding="utf-8-sig",
)

print("Tabla definitiva guardada en:")
print(validated_path)

Tabla definitiva guardada en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\portal_subzone_mapping_validated.csv


In [110]:
# Las correspondencias basadas en una coincidencia única
# de barrio oficial dentro del mismo distrito se consideran
# validadas mediante una regla reproducible.

automatic_validation_mask = (
    mapping_validated["mapping_status"]
    == "proposed_high_confidence"
)

mapping_validated.loc[
    automatic_validation_mask,
    "mapping_status",
] = "validated_rule"

print(
    "Correspondencias validadas mediante regla:",
    automatic_validation_mask.sum(),
)

print(
    mapping_validated["mapping_status"].value_counts()
)

Correspondencias validadas mediante regla: 19
mapping_status
validated_rule    19
validated         13
Name: count, dtype: int64


In [111]:
mapping_validated.to_csv(
    validated_path,
    index=False,
    encoding="utf-8-sig",
)

print("Tabla definitiva actualizada:")
print(validated_path)

Tabla definitiva actualizada:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\portal_subzone_mapping_validated.csv


### Aplicar la tabla a los 11.826 anuncios

In [112]:
# Preparamos únicamente las columnas necesarias para aplicar
# las equivalencias territoriales a los anuncios.

mapping_for_merge = mapping_validated[
    [
        "district_name_official",
        "subzona_portal_norm",
        "selected_official_neighborhood",
        "selected_zone_key",
        "mapping_status",
        "validation_method",
        "confidence",
        "source_name",
        "source_reference",
        "validation_notes",
    ]
].copy()

# Aplicamos la tabla a los anuncios.
# many_to_one garantiza que cada subzona tenga una sola decisión.

listings_geocoded = df_matched.merge(
    mapping_for_merge,
    on=[
        "district_name_official",
        "subzona_portal_norm",
    ],
    how="left",
    validate="many_to_one",
)

print("Anuncios antes del merge:", len(df_matched))
print("Anuncios después del merge:", len(listings_geocoded))

Anuncios antes del merge: 11826
Anuncios después del merge: 11826


### Crear las columnas territoriales definitivas

In [113]:
# Conservamos las asignaciones exactas existentes y,
# cuando no existen, utilizamos las equivalencias validadas.

listings_geocoded["zone_key_final"] = (
    listings_geocoded["zone_key"]
    .combine_first(
        listings_geocoded["selected_zone_key"]
    )
)

listings_geocoded["neighborhood_name_final"] = (
    listings_geocoded["neighborhood_name"]
    .combine_first(
        listings_geocoded[
            "selected_official_neighborhood"
        ]
    )
)

listings_geocoded["district_name_final"] = (
    listings_geocoded["district_name"]
    .combine_first(
        listings_geocoded["district_name_official"]
    )
)

In [114]:
# ============================================================
# Método, confianza y trazabilidad territorial
# ============================================================

exact_match_mask = (
    listings_geocoded["zone_key"].notna()
)

validated_mapping_mask = (
    listings_geocoded["zone_key"].isna()
    & listings_geocoded[
        "selected_zone_key"
    ].notna()
)

district_only_mask = (
    listings_geocoded[
        "zone_key_final"
    ].isna()
    & listings_geocoded[
        "district_name_final"
    ].notna()
)


# Añadimos trazabilidad también a las coincidencias exactas.

listings_geocoded.loc[
    exact_match_mask,
    "mapping_status",
] = "validated_exact"

listings_geocoded.loc[
    exact_match_mask,
    "validation_method",
] = "exact_name_same_district"

listings_geocoded.loc[
    exact_match_mask,
    "confidence",
] = "high"

listings_geocoded.loc[
    exact_match_mask,
    "source_name",
] = "División oficial de barrios de Madrid"

listings_geocoded.loc[
    exact_match_mask,
    "source_reference",
] = "data/processed/zones_master.csv"

listings_geocoded.loc[
    exact_match_mask,
    "validation_notes",
] = (
    "Coincidencia exacta entre la subzona normalizada "
    "del portal y el nombre del barrio oficial, "
    "perteneciendo ambos al mismo distrito."
)


listings_geocoded[
    "mapping_method_final"
] = np.select(
    [
        exact_match_mask,
        validated_mapping_mask,
        district_only_mask,
    ],
    [
        "exact_name_same_district",
        "validated_subzone_mapping",
        "district_only",
    ],
    default="unresolved",
)


listings_geocoded[
    "mapping_detail_final"
] = np.where(
    exact_match_mask,
    "exact_name_same_district",
    listings_geocoded[
        "validation_method"
    ],
)


listings_geocoded[
    "mapping_confidence_final"
] = np.select(
    [
        exact_match_mask,
        validated_mapping_mask,
        district_only_mask,
    ],
    [
        "high",
        listings_geocoded[
            "confidence"
        ].fillna("unknown"),
        "district_only",
    ],
    default="unresolved",
)

### Validar el resultado territorial

In [115]:
total_listings = len(listings_geocoded)

assigned_neighborhood = (
    listings_geocoded["zone_key_final"]
    .notna()
    .sum()
)

district_only_listings = (
    listings_geocoded["mapping_method_final"]
    .eq("district_only")
    .sum()
)

unresolved_listings = (
    listings_geocoded["mapping_method_final"]
    .eq("unresolved")
    .sum()
)

coverage_percentage = (
    assigned_neighborhood
    / total_listings
    * 100
)

print("Total de anuncios:", total_listings)
print(
    "Anuncios asignados a barrio:",
    assigned_neighborhood,
)
print(
    "Anuncios conservados solo a nivel de distrito:",
    district_only_listings,
)
print(
    "Anuncios completamente sin resolver:",
    unresolved_listings,
)
print(
    "Cobertura final a nivel de barrio:",
    round(coverage_percentage, 2),
    "%",
)

display(
    listings_geocoded[
        "mapping_method_final"
    ].value_counts()
)

Total de anuncios: 11826
Anuncios asignados a barrio: 11826
Anuncios conservados solo a nivel de distrito: 0
Anuncios completamente sin resolver: 0
Cobertura final a nivel de barrio: 100.0 %


mapping_method_final
exact_name_same_district     9002
validated_subzone_mapping    2824
Name: count, dtype: int64

### Guardar el resultado territorial

In [116]:
geocoded_path = (
    INTERIM_DIR
    / "listings_geocoded.csv"
)

listings_geocoded.to_csv(
    geocoded_path,
    index=False,
    encoding="utf-8-sig",
)

print("Dataset territorial guardado en:")
print(geocoded_path)

Dataset territorial guardado en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\listings_geocoded.csv


### Crear una copia de trabajo

In [117]:
# Creamos una copia para la limpieza.
# listings_geocoded se conserva sin modificar.

listings_cleaning = listings_geocoded.copy()

print("Anuncios al comenzar la limpieza:", len(listings_cleaning))

Anuncios al comenzar la limpieza: 11826


### Convertir precio y superficie a formato numérico

In [118]:
numeric_columns = [
    "PrecioActual",
    "PrecioAnterior",
    "metros",
]

for column in numeric_columns:
    listings_cleaning[f"{column}_numeric"] = pd.to_numeric(
        listings_cleaning[column],
        errors="coerce",
    )

conversion_summary = pd.DataFrame(
    {
        "variable": numeric_columns,
        "nulos_originales": [
            listings_cleaning[column].isna().sum()
            for column in numeric_columns
        ],
        "nulos_despues_conversion": [
            listings_cleaning[f"{column}_numeric"].isna().sum()
            for column in numeric_columns
        ],
    }
)

conversion_summary["errores_conversion"] = (
    conversion_summary["nulos_despues_conversion"]
    - conversion_summary["nulos_originales"]
)

display(conversion_summary)

,variable,nulos_originales,nulos_despues_conversion,errores_conversion
0,PrecioActual,0,0,0
1,PrecioAnterior,0,0,0
2,metros,0,0,0


### Crear nombres claros para las variables limpias

In [119]:
listings_cleaning["price_eur"] = (
    listings_cleaning["PrecioActual_numeric"]
)

listings_cleaning["previous_price_eur"] = (
    listings_cleaning["PrecioAnterior_numeric"]
)

listings_cleaning["area_m2"] = (
    listings_cleaning["metros_numeric"]
)

### Tratar los ceros de PrecioAnterior

In [120]:
listings_cleaning["previous_price_available"] = (
    listings_cleaning["previous_price_eur"].gt(0)
)

listings_cleaning.loc[
    listings_cleaning["previous_price_eur"].le(0),
    "previous_price_eur",
] = np.nan

print(
    "Anuncios con precio anterior disponible:",
    listings_cleaning[
        "previous_price_available"
    ].sum(),
)

print(
    "Anuncios sin precio anterior disponible:",
    listings_cleaning[
        "previous_price_available"
    ].eq(False).sum(),
)

Anuncios con precio anterior disponible: 1113
Anuncios sin precio anterior disponible: 10713


### Marcar valores físicamente inválidos

In [121]:
listings_cleaning["invalid_price_basic"] = (
    listings_cleaning["price_eur"].isna()
    | listings_cleaning["price_eur"].le(0)
)

listings_cleaning["invalid_area_basic"] = (
    listings_cleaning["area_m2"].isna()
    | listings_cleaning["area_m2"].le(0)
)

print(
    "Precios ausentes, cero o negativos:",
    listings_cleaning["invalid_price_basic"].sum(),
)

print(
    "Superficies ausentes, cero o negativas:",
    listings_cleaning["invalid_area_basic"].sum(),
)

Precios ausentes, cero o negativos: 0
Superficies ausentes, cero o negativas: 0


### Calcular el precio por metro cuadrado

In [122]:
valid_price_area_mask = (
    ~listings_cleaning["invalid_price_basic"]
    & ~listings_cleaning["invalid_area_basic"]
)

listings_cleaning["price_per_m2"] = np.nan

listings_cleaning.loc[
    valid_price_area_mask,
    "price_per_m2",
] = (
    listings_cleaning.loc[
        valid_price_area_mask,
        "price_eur",
    ]
    / listings_cleaning.loc[
        valid_price_area_mask,
        "area_m2",
    ]
)

print(
    "Anuncios con precio por m² calculable:",
    listings_cleaning["price_per_m2"].notna().sum(),
)

Anuncios con precio por m² calculable: 11826


### Mostrar la auditoría estadística

In [123]:
variables_auditadas = [
    "price_eur",
    "area_m2",
    "price_per_m2",
]

audit_price_area = (
    listings_cleaning[variables_auditadas]
    .describe(
        percentiles=[
            0.001,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            0.999,
        ]
    )
    .T
)

display(audit_price_area)

,count,mean,std,min,0.1%,1%,5%,25%,50%,75%,95%,99%,99.9%,max
price_eur,11826.0,1.030501e+06,1.237718e+06,12000.00,55500.000000,86075.00000,148625.000000,289000.000000,620000.0,1.329000e+06,3.300000e+06,6.000000e+06,1.208750e+07,2.300000e+07
area_m2,11826.0,1.537900e+02,7.662177e+02,11.00,16.825000,26.00000,40.000000,68.000000,103.0,1.600000e+02,3.887500e+02,7.500000e+02,1.575000e+03,7.500000e+04
price_per_m2,11826.0,6.654731e+03,3.581972e+03,2.56,814.335812,1363.30505,2233.333333,3909.090909,6000.0,8.725490e+03,1.329436e+04,1.715816e+04,2.411469e+04,3.364486e+04


### Ver los valores extremos

In [124]:
print("Anuncios con menor precio positivo:")
display(
    listings_cleaning.loc[
        listings_cleaning["price_eur"].gt(0),
        [
            "titulo",
            "price_eur",
            "area_m2",
            "price_per_m2",
            "neighborhood_name_final",
            "Enlace",
        ],
    ]
    .sort_values("price_eur")
    .head(20)
)

print("Anuncios con mayor precio:")
display(
    listings_cleaning.loc[
        listings_cleaning["price_eur"].notna(),
        [
            "titulo",
            "price_eur",
            "area_m2",
            "price_per_m2",
            "neighborhood_name_final",
            "Enlace",
        ],
    ]
    .sort_values("price_eur", ascending=False)
    .head(20)
)

print("Superficies más pequeñas:")
display(
    listings_cleaning.loc[
        listings_cleaning["area_m2"].gt(0),
        [
            "titulo",
            "price_eur",
            "area_m2",
            "price_per_m2",
            "neighborhood_name_final",
            "Enlace",
        ],
    ]
    .sort_values("area_m2")
    .head(20)
)

print("Superficies más grandes:")
display(
    listings_cleaning.loc[
        listings_cleaning["area_m2"].notna(),
        [
            "titulo",
            "price_eur",
            "area_m2",
            "price_per_m2",
            "neighborhood_name_final",
            "Enlace",
        ],
    ]
    .sort_values("area_m2", ascending=False)
    .head(20)
)

Anuncios con menor precio positivo:


,titulo,price_eur,area_m2,price_per_m2,neighborhood_name_final,Enlace
6669,"Piso en calle Marmolistas, Arcos, Madrid",12000,61,196.721311,Arcos,https://www.idealista.com/inmueble/106859553/
3683,"Piso en calle Marmolistas, Arcos, Madrid",12000,61,196.721311,Arcos,https://www.idealista.com/inmueble/107240309/
691,"Piso en calle del Pico Cebollera, San Diego, Madrid",33500,51,656.862745,San Diego,https://www.idealista.com/inmueble/106987531/
9717,"Piso en calle Del Puerto de Panticosa, 6, Numancia, Madrid",35000,20,1750.000000,Numancia,https://www.idealista.com/inmueble/99812663/
2044,"Piso en calle de San Dalmacio, Villaverde Alto, Madrid",49900,60,831.666667,Villaverde Alto - Casco Histórico de Villaverde,https://www.idealista.com/inmueble/36146834/
978,"Piso en calle de San Dalmacio, Villaverde Alto, Madrid",49900,61,818.032787,Villaverde Alto - Casco Histórico de Villaverde,https://www.idealista.com/inmueble/104824996/
6447,"Piso en calle de Sierra Elvira, 32, Numancia, Madrid",50000,22,2272.727273,Numancia,https://www.idealista.com/inmueble/107287342/
8212,"Piso en calle de Sierra Elvira, 32, Numancia, Madrid",50000,22,2272.727273,Numancia,https://www.idealista.com/inmueble/81408177/
11636,"Piso en calle Emilio Raboso, San Diego, Madrid",52400,27,1940.740741,San Diego,https://www.idealista.com/inmueble/104891274/
11116,"Piso en calle Concepción de la Oliva, s/n, Butarque, Madrid",54000,54,1000.000000,Butarque,https://www.idealista.com/inmueble/105751198/


Anuncios con mayor precio:


,titulo,price_eur,area_m2,price_per_m2,neighborhood_name_final,Enlace
6489,"Casa o chalet independiente en calle de Castellón de la Plana, El Viso, Madrid",23000000,1000,23000.000000,El Viso,https://www.idealista.com/inmueble/107095805/
10258,"Ático en plaza de la Independencia, Jerónimos, Madrid",18000000,535,33644.859813,Los Jerónimos,https://www.idealista.com/inmueble/105924056/
5735,"Ático en El Viso, Madrid",17500000,697,25107.604017,El Viso,https://www.idealista.com/inmueble/106795928/
11723,"Piso en calle de Arrieta, Palacio, Madrid",15000000,666,22522.522523,Palacio,https://www.idealista.com/inmueble/106035767/
1914,"Dúplex en plaza Marqués de Salamanca, 11, Lista, Madrid",13090000,605,21636.363636,Lista,https://www.idealista.com/inmueble/101733694/
7614,"Piso en Recoletos, Madrid",13000000,421,30878.859857,Recoletos,https://www.idealista.com/inmueble/107165396/
6546,"Piso en calle Velázquez, Recoletos, Madrid",13000000,425,30588.235294,Recoletos,https://www.idealista.com/inmueble/107166409/
6191,"Piso en calle de Velázquez, Recoletos, Madrid",13000000,421,30878.859857,Recoletos,https://www.idealista.com/inmueble/106652699/
7522,"Piso en calle Velázquez, Recoletos, Madrid",13000000,421,30878.859857,Recoletos,https://www.idealista.com/inmueble/107212046/
9675,"Piso en Recoletos, Madrid",13000000,421,30878.859857,Recoletos,https://www.idealista.com/inmueble/106771279/


Superficies más pequeñas:


,titulo,price_eur,area_m2,price_per_m2,neighborhood_name_final,Enlace
407,"Ático en calle de Juanelo, Lavapiés-Embajadores, Madrid",125000,11,11363.636364,Embajadores,https://www.idealista.com/inmueble/104305109/
6779,"Estudio en paseo de las Acacias, 9, Acacias, Madrid",108000,12,9000.000000,Acacias,https://www.idealista.com/inmueble/106681153/
3219,"Piso en calle San Cosme y San Damián, Lavapiés-Embajadores, Madrid",129900,12,10825.000000,Embajadores,https://www.idealista.com/inmueble/107260880/
11743,"Estudio en paseo de las Acacias, Acacias, Madrid",108000,13,8307.692308,Acacias,https://www.idealista.com/inmueble/106983822/
7917,"Piso en paseo de las Acacias, Acacias, Madrid",108000,13,8307.692308,Acacias,https://www.idealista.com/inmueble/107145180/
10531,"Estudio en paseo de las Acacias, Acacias, Madrid",108000,13,8307.692308,Acacias,https://www.idealista.com/inmueble/107058209/
1421,"Estudio en paseo de las Acacias, Acacias, Madrid",108000,15,7200.000000,Acacias,https://www.idealista.com/inmueble/107146672/
3549,"Estudio en paseo de las Acacias, Acacias, Madrid",108000,15,7200.000000,Acacias,https://www.idealista.com/inmueble/106590649/
4897,"Estudio en Acacias, Madrid",108000,15,7200.000000,Acacias,https://www.idealista.com/inmueble/107304721/
7902,"Estudio en paseo de las Acacias, Acacias, Madrid",108000,15,7200.000000,Acacias,https://www.idealista.com/inmueble/107013373/


Superficies más grandes:


,titulo,price_eur,area_m2,price_per_m2,neighborhood_name_final,Enlace
6979,"Piso en calle de Emilio Ferrari, s/n, Pueblo Nuevo, Madrid",192000,75000,2.560000,Pueblo Nuevo,https://www.idealista.com/inmueble/107196960/
7997,"Chalet adosado en Palomas, Madrid",1495000,33175,45.064054,Palomas,https://www.idealista.com/inmueble/106105382/
8284,"Casa o chalet independiente en Ciudad Universitaria, Madrid",8250000,3015,2736.318408,Ciudad Universitaria,https://www.idealista.com/inmueble/100473157/
699,"Casa o chalet independiente en El Plantío, Madrid",2500000,2500,1000.000000,El Plantío,https://www.idealista.com/inmueble/107249503/
6225,"Casa o chalet independiente en Amorabieta, El Plantío, Madrid",2900000,1768,1640.271493,El Plantío,https://www.idealista.com/inmueble/104179896/
192,"Casa o chalet independiente en calle de Amorebieta, El Plantío, Madrid",2900000,1768,1640.271493,El Plantío,https://www.idealista.com/inmueble/103994353/
7807,"Casa o chalet independiente en calle de Amorebieta, El Plantío, Madrid",2900000,1768,1640.271493,El Plantío,https://www.idealista.com/inmueble/103628504/
7958,"Casa o chalet independiente en calle Eibar, El Plantío, Madrid",9000000,1674,5376.344086,El Plantío,https://www.idealista.com/inmueble/106713206/
6035,"Casa o chalet independiente en calle Eibar, El Plantío, Madrid",9000000,1674,5376.344086,El Plantío,https://www.idealista.com/inmueble/106751540/
341,"Casa o chalet independiente en calle Eibar, El Plantío, Madrid",9000000,1674,5376.344086,El Plantío,https://www.idealista.com/inmueble/105615858/


### Identificación de valores extremos para revisión

In [125]:
variables_extremos = [
    "price_eur",
    "area_m2",
    "price_per_m2",
]

limites_extremos = {}

for variable in variables_extremos:
    limites_extremos[variable] = {
        "inferior": listings_cleaning[variable].quantile(0.001),
        "superior": listings_cleaning[variable].quantile(0.999),
    }

limites_extremos_df = pd.DataFrame(limites_extremos).T

display(limites_extremos_df)

,inferior,superior
price_eur,55500.000000,1.208750e+07
area_m2,16.825000,1.575000e+03
price_per_m2,814.335812,2.411469e+04


### Crear indicadores de revisión

In [126]:
listings_cleaning["flag_extreme_low_price"] = (
    listings_cleaning["price_eur"]
    < limites_extremos["price_eur"]["inferior"]
)

listings_cleaning["flag_extreme_high_price"] = (
    listings_cleaning["price_eur"]
    > limites_extremos["price_eur"]["superior"]
)

listings_cleaning["flag_extreme_low_area"] = (
    listings_cleaning["area_m2"]
    < limites_extremos["area_m2"]["inferior"]
)

listings_cleaning["flag_extreme_high_area"] = (
    listings_cleaning["area_m2"]
    > limites_extremos["area_m2"]["superior"]
)

listings_cleaning["flag_extreme_low_price_m2"] = (
    listings_cleaning["price_per_m2"]
    < limites_extremos["price_per_m2"]["inferior"]
)

listings_cleaning["flag_extreme_high_price_m2"] = (
    listings_cleaning["price_per_m2"]
    > limites_extremos["price_per_m2"]["superior"]
)

extreme_flag_columns = [
    "flag_extreme_low_price",
    "flag_extreme_high_price",
    "flag_extreme_low_area",
    "flag_extreme_high_area",
    "flag_extreme_low_price_m2",
    "flag_extreme_high_price_m2",
]

listings_cleaning["requires_price_area_review"] = (
    listings_cleaning[extreme_flag_columns]
    .any(axis=1)
)

print(
    "Anuncios candidatos a revisión:",
    listings_cleaning["requires_price_area_review"].sum(),
)

display(
    listings_cleaning[
        extreme_flag_columns
    ].sum()
)

Anuncios candidatos a revisión: 58


flag_extreme_low_price        10
flag_extreme_high_price       12
flag_extreme_low_area         12
flag_extreme_high_area        12
flag_extreme_low_price_m2     12
flag_extreme_high_price_m2    12
dtype: int64

### Registrar el motivo de revisión

In [127]:
flag_labels = {
    "flag_extreme_low_price": "precio_muy_bajo",
    "flag_extreme_high_price": "precio_muy_alto",
    "flag_extreme_low_area": "superficie_muy_baja",
    "flag_extreme_high_area": "superficie_muy_alta",
    "flag_extreme_low_price_m2": "precio_m2_muy_bajo",
    "flag_extreme_high_price_m2": "precio_m2_muy_alto",
}


def obtener_motivos_revision(fila):
    motivos = [
        etiqueta
        for columna, etiqueta in flag_labels.items()
        if fila[columna]
    ]

    return ";".join(motivos)


listings_cleaning["price_area_review_reason"] = (
    listings_cleaning.apply(
        obtener_motivos_revision,
        axis=1,
    )
)

### Crear y guardar la tabla de revisión

In [128]:
price_area_review = listings_cleaning.loc[
    listings_cleaning["requires_price_area_review"],
    [
        "titulo",
        "price_eur",
        "area_m2",
        "price_per_m2",
        "habitaciones",
        "baños",
        "tags",
        "descripcion",
        "neighborhood_name_final",
        "price_area_review_reason",
        "Enlace",
    ],
].copy()

price_area_review = price_area_review.sort_values(
    [
        "price_area_review_reason",
        "price_per_m2",
    ]
)

display(price_area_review)

price_area_review_path = (
    INTERIM_DIR
    / "price_area_anomaly_candidates.csv"
)

price_area_review.to_csv(
    price_area_review_path,
    index=False,
    encoding="utf-8-sig",
)

print("Tabla de revisión guardada en:")
print(price_area_review_path)

,titulo,price_eur,area_m2,price_per_m2,habitaciones,baños,tags,descripcion,neighborhood_name_final,price_area_review_reason,Enlace
5587,"Ático en calle de José Ortega y Gasset, Castellana, Madrid",8000000,330,24242.424242,3.0,0,"ÁTICO,LUJO,INMOBILIARIA,IMPRESIONANTE","Álvora Capital Properties, consultora inmobiliaria de alta calidad en la gestión de activos prime en Madrid, les presenta esté impresionante ático...",Castellana,precio_m2_muy_alto,https://www.idealista.com/inmueble/107198854/
3108,"Ático en calle de José Ortega y Gasset, Castellana, Madrid",8000000,326,24539.877301,3.0,0,"TERRAZA,ÁTICO,REFORMAR",Olivares Consultores vende ático a reformar en el barrio de Salamanca. El edificio es de fachada clásica. Dispone de terraza de 44m2. Su ubicación...,Castellana,precio_m2_muy_alto,https://www.idealista.com/inmueble/107144247/
2429,"Ático en Recoletos, Madrid",3990000,160,24937.500000,3.0,0,"VIVIENDA,REFORMADO,ÁTICO,LUJO,ELEGANTE","Descubre la exquisitez de este increíble ático reformado, ubicado en el distinguido barrio de Recoletos. Con una distribución elegante y cuidadosa...",Recoletos,precio_m2_muy_alto,https://www.idealista.com/inmueble/106994296/
5314,"Ático en Castellana, Madrid",8700000,330,26363.636364,3.0,0,"VIVIENDA,TERRAZA,AMPLIO,ÁTICO,ESPECTACULAR,REFORMADA,ESTRENAR,PRESTIGIOSO,ESTUDIO",Espectacular ático con terraza de 45 mt y orientación sur.La vivienda se vende totalmente reformada a estrenar. En este momento se encuentra en pr...,Castellana,precio_m2_muy_alto,https://www.idealista.com/inmueble/107152772/
6053,"Ático en Recoletos, Madrid",5250000,160,32812.500000,3.0,0,"REFORMADO,TERRAZA,ÁTICO,IMPRESIONANTE,PRESTIGIOSO,METRO","Moro Real Estate presenta este impresionante ático, recién reformado, situado en una de las zonas más codiciadas del Barrio de Salamanca, a escaso...",Recoletos,precio_m2_muy_alto,https://www.idealista.com/inmueble/106137517/
2714,"Estudio en San Diego, Madrid",114500,811,141.183724,NaN,0,"PISO,FUNCIONAL","Funcional piso en Madrid, ideal para parejas. El inmueble consta de 811 m² construidos distribuidos en salón-comedor, cocina. Cuenta con todos los...",San Diego,precio_m2_muy_bajo,https://www.idealista.com/inmueble/107028561/
7750,"Piso en Entrevías, Madrid",161000,479,336.116910,3.0,1,"VIVIENDA,PISO,LUMINOSO,HOGAR,FUNCIONAL","Presentamos este encantador piso de segunda mano en Madrid, ideal para familias que buscan un hogar acogedor y bien ubicado. Con una superficie de...",Entrevías,precio_m2_muy_bajo,https://www.idealista.com/inmueble/101241888/
7283,"Piso en calle de la Unanimidad, Los Rosales, Madrid",78900,103,766.019417,3.0,0,"VIVIENDA,PISO","Piso de 103,76 m² situado en la provincia de Madrid. Esta vivienda de segunda mano está distribuida en varias dependencias y servicios. Dispone de...",Los Rosales,precio_m2_muy_bajo,https://www.idealista.com/inmueble/106658093/
3177,"Piso en calle de Beniferri, San Cristóbal, Madrid",86300,112,770.535714,3.0,0,"VIVIENDA,PISO",Vivienda piso en la calle Beniferri de Madrid. Tiene una superficie aproximada de 80 m2. - El inmueble se encuentra ocupado ilegalmente por tercer...,San Cristóbal,precio_m2_muy_bajo,https://www.idealista.com/inmueble/106563601/
7691,"Piso en calle de Beniferri, 32, San Cristóbal, Madrid",86300,112,770.535714,3.0,0,NaN,"*** OKUPADA, SE VENDE LA PROPIEDAD., NO L APOSESION.El inmueble se encuentra ocupado ilegalmente por terceros y sin los suministros debidamente co...",San Cristóbal,precio_m2_muy_bajo,https://www.idealista.com/inmueble/107151576/


Tabla de revisión guardada en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\price_area_anomaly_candidates.csv


### reglas de revisión

In [129]:
import re
import unicodedata


def normalizar_texto_reglas(valor):
    """
    Normaliza el texto para aplicar reglas reproducibles:
    minúsculas, eliminación de tildes y espacios repetidos.
    """
    if pd.isna(valor):
        return ""

    texto = str(valor).lower()

    texto = unicodedata.normalize(
        "NFKD",
        texto,
    ).encode(
        "ascii",
        "ignore",
    ).decode("utf-8")

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


price_area_reviewed = price_area_review.copy()

price_area_reviewed["combined_text_norm"] = (
    price_area_reviewed["titulo"].fillna("")
    + " "
    + price_area_reviewed["tags"].fillna("")
    + " "
    + price_area_reviewed["descripcion"].fillna("")
).apply(normalizar_texto_reglas)

### Identificar operaciones especiales

In [130]:
price_area_reviewed["flag_occupied_or_no_possession"] = (
    price_area_reviewed["combined_text_norm"].str.contains(
        (
            r"ocupad|okupad|sin posesion|"
            r"venta sin posesion|no se puede visitar"
        ),
        regex=True,
    )
)

price_area_reviewed["flag_partial_ownership"] = (
    price_area_reviewed["combined_text_norm"].str.contains(
        (
            r"proindiviso|solo una parte de la propiedad|"
            r"parte de la propiedad"
        ),
        regex=True,
    )
)

price_area_reviewed["flag_auction"] = (
    price_area_reviewed["combined_text_norm"].str.contains(
        r"subasta",
        regex=True,
    )
)

price_area_reviewed["flag_special_transaction"] = (
    price_area_reviewed[
        [
            "flag_occupied_or_no_possession",
            "flag_partial_ownership",
            "flag_auction",
        ]
    ].any(axis=1)
)

### Identificar errores probables de superficie

In [131]:
# Esta regla no corrige automáticamente la superficie.
# Solo identifica registros incompatibles con el tipo
# y el precio declarado.

price_area_reviewed["flag_probable_area_error"] = (
    price_area_reviewed["area_m2"].ge(10000)
    | (
        price_area_reviewed["area_m2"].gt(400)
        & price_area_reviewed["price_per_m2"].lt(500)
    )
)

price_area_reviewed["flag_non_standard_use"] = (
    price_area_reviewed["combined_text_norm"].str.contains(
        (
            r"local o trastero|"
            r"se puede utilizar de local|"
            r"uso como local"
        ),
        regex=True,
    )
)

### Registrar la decisión metodológica

In [132]:
price_area_reviewed["recommended_action"] = np.select(
    [
        price_area_reviewed["flag_probable_area_error"],
        price_area_reviewed["flag_special_transaction"],
        price_area_reviewed["flag_non_standard_use"],
    ],
    [
        "exclude_probable_area_error",
        "exclude_special_transaction",
        "review_non_standard_use",
    ],
    default="retain_extreme_plausible",
)

price_area_reviewed["review_notes"] = np.select(
    [
        price_area_reviewed["flag_probable_area_error"],
        price_area_reviewed["flag_special_transaction"],
        price_area_reviewed["flag_non_standard_use"],
    ],
    [
        (
            "Superficie incompatible con el tipo de inmueble "
            "o con el precio declarado. No se corrige sin una "
            "fuente verificable."
        ),
        (
            "Operación no comparable con una compraventa "
            "residencial convencional."
        ),
        (
            "El anuncio indica un posible uso distinto "
            "del residencial."
        ),
    ],
    default=(
        "Valor extremo, pero compatible con la descripción "
        "del inmueble."
    ),
)

display(
    price_area_reviewed[
        "recommended_action"
    ].value_counts()
)

recommended_action
retain_extreme_plausible       42
exclude_special_transaction    11
exclude_probable_area_error     4
review_non_standard_use         1
Name: count, dtype: int64

### Ver los cuatro errores probables

In [133]:
display(
    price_area_reviewed.loc[
        price_area_reviewed["flag_probable_area_error"],
        [
            "titulo",
            "price_eur",
            "area_m2",
            "price_per_m2",
            "flag_special_transaction",
            "recommended_action",
        ],
    ]
)

,titulo,price_eur,area_m2,price_per_m2,flag_special_transaction,recommended_action
2714,"Estudio en San Diego, Madrid",114500,811,141.183724,False,exclude_probable_area_error
7750,"Piso en Entrevías, Madrid",161000,479,336.116910,False,exclude_probable_area_error
6979,"Piso en calle de Emilio Ferrari, s/n, Pueblo Nuevo, Madrid",192000,75000,2.560000,True,exclude_probable_area_error
7997,"Chalet adosado en Palomas, Madrid",1495000,33175,45.064054,False,exclude_probable_area_error


### Guardar la revisión

In [134]:
reviewed_anomalies_path = (
    INTERIM_DIR
    / "price_area_anomaly_reviewed.csv"
)

price_area_reviewed.to_csv(
    reviewed_anomalies_path,
    index=False,
    encoding="utf-8-sig",
)

print("Revisión de anomalías guardada en:")
print(reviewed_anomalies_path)

Revisión de anomalías guardada en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\price_area_anomaly_reviewed.csv


### revisión al dataset completo

In [135]:
# ============================================================
# Incorporación de la revisión al dataset completo
# ============================================================

review_for_merge = price_area_reviewed[
    [
        "Enlace",
        "recommended_action",
        "review_notes",
        "flag_probable_area_error",
        "flag_special_transaction",
        "flag_non_standard_use",
    ]
].copy()

# Evitamos nombres ambiguos al incorporarlo al dataset completo.
review_for_merge = review_for_merge.rename(
    columns={
        "recommended_action": "price_area_action",
        "review_notes": "price_area_action_notes",
        "flag_probable_area_error": (
            "flag_probable_area_error_reviewed"
        ),
        "flag_special_transaction": (
            "flag_special_transaction_reviewed"
        ),
        "flag_non_standard_use": (
            "flag_non_standard_use_reviewed"
        ),
    }
)

if review_for_merge["Enlace"].duplicated().any():
    raise ValueError(
        "Existen enlaces duplicados en la tabla de revisión."
    )

listings_cleaning = listings_cleaning.merge(
    review_for_merge,
    on="Enlace",
    how="left",
    validate="one_to_one",
)

listings_cleaning["price_area_action"] = (
    listings_cleaning["price_area_action"]
    .fillna("not_extreme")
)

listings_cleaning["price_area_action_notes"] = (
    listings_cleaning["price_area_action_notes"]
    .fillna(
        "El anuncio no fue seleccionado como extremo "
        "en la auditoría de precio y superficie."
    )
)

reviewed_flag_columns = [
    "flag_probable_area_error_reviewed",
    "flag_special_transaction_reviewed",
    "flag_non_standard_use_reviewed",
]

for column in reviewed_flag_columns:
    listings_cleaning[column] = (
        listings_cleaning[column]
        .fillna(False)
        .astype(bool)
    )

display(
    listings_cleaning[
        "price_area_action"
    ].value_counts()
)

price_area_action
not_extreme                    11768
retain_extreme_plausible          42
exclude_special_transaction       11
exclude_probable_area_error        4
review_non_standard_use            1
Name: count, dtype: int64

### Comprobar que no se ha perdido ningún anuncio

In [136]:
print(
    "Total de anuncios después de incorporar la revisión:",
    len(listings_cleaning),
)

print(
    "Anuncios sin decisión:",
    listings_cleaning[
        "price_area_action"
    ].isna().sum(),
)

print(
    "Enlaces duplicados:",
    listings_cleaning[
        "Enlace"
    ].duplicated().sum(),
)

Total de anuncios después de incorporar la revisión: 11826
Anuncios sin decisión: 0
Enlaces duplicados: 0


### buscar operaciones especiales en toda la base

In [137]:
# ============================================================
# Detección de operaciones especiales en los 11.826 anuncios
# ============================================================

# Unimos título, etiquetas y descripción.
# El texto se normaliza para eliminar diferencias de mayúsculas,
# tildes, signos y otros elementos que dificultan la búsqueda.

listings_cleaning["combined_text_norm_all"] = (
    listings_cleaning["titulo"].fillna("")
    + " "
    + listings_cleaning["tags"].fillna("")
    + " "
    + listings_cleaning["descripcion"].fillna("")
).apply(normalizar_texto_reglas)


# Los límites \b evitan coincidencias incorrectas.
# Por ejemplo, "ocupad" podría coincidir dentro de "desocupado",
# mientras que \bocupad[ao]s?\b solo encuentra ocupado/ocupada.

special_transaction_patterns = {
    "occupied_or_no_possession": (
    r"\bocupad[ao]s?\b|"
    r"\bokupad[ao]s?\b|"
    r"\bocupacion ilegal\b|"
    r"\bsin posesion\b|"
    r"\bventa sin posesion\b"
),

    "bare_ownership_or_usufruct": (
        r"\bnuda propiedad\b|"
        r"\busufruct(?:o|uario|uaria|uarios|uarias)?\b"
    ),

    "partial_ownership": (
        r"\bproindiviso\b|"
        r"\bparte indivisa\b|"
        r"\bparticipacion indivisa\b|"
        r"\bparte de la propiedad\b|"
        r"\bsolo una parte de la propiedad\b"
    ),

    "auction": (
        r"\bsubasta\b|"
        r"\bcesion de remate\b"
    ),

    "tenant_or_rented": (
        r"\bcon inquilin[oa]s?\b|"
        r"\bse vende alquilad[ao]\b|"
        r"\bactualmente alquilad[ao]\b|"
        r"\bse vende arrendad[ao]\b|"
        r"\bactualmente arrendad[ao]\b|"
        r"\bcontrato de alquiler vigente\b|"
        r"\bcontrato de arrendamiento vigente\b|"
        r"\bcon contrato de alquiler\b"
    ),
}


# Creamos una columna booleana por cada categoría.

for category, pattern in special_transaction_patterns.items():
    flag_column = f"flag_special_{category}"

    listings_cleaning[flag_column] = (
        listings_cleaning["combined_text_norm_all"]
        .str.contains(
            pattern,
            regex=True,
            na=False,
        )
    )


global_special_flag_columns = [
    f"flag_special_{category}"
    for category in special_transaction_patterns
]


# Una operación es especial si cumple al menos una categoría.

listings_cleaning["flag_special_transaction_all"] = (
    listings_cleaning[
        global_special_flag_columns
    ].any(axis=1)
)


# Guardamos también la razón concreta de la detección.
# Esto permite explicar por qué se excluye cada anuncio.

def obtain_special_transaction_reasons(row):
    reasons = []

    for category in special_transaction_patterns:
        flag_column = f"flag_special_{category}"

        if row[flag_column]:
            reasons.append(category)

    return "|".join(reasons)


listings_cleaning["special_transaction_reasons"] = (
    listings_cleaning.apply(
        obtain_special_transaction_reasons,
        axis=1,
    )
)


print(
    "Operaciones especiales detectadas en toda la base:",
    listings_cleaning[
        "flag_special_transaction_all"
    ].sum(),
)

print("\nNúmero de detecciones por categoría:")

display(
    listings_cleaning[
        global_special_flag_columns
    ].sum()
)

Operaciones especiales detectadas en toda la base: 619

Número de detecciones por categoría:


flag_special_occupied_or_no_possession     358
flag_special_bare_ownership_or_usufruct     82
flag_special_partial_ownership               7
flag_special_auction                        30
flag_special_tenant_or_rented              147
dtype: int64

### Guardar los candidatos para revisarlos

In [138]:
special_transaction_candidates = listings_cleaning.loc[
    listings_cleaning["flag_special_transaction_all"],
    [
        "titulo",
        "price_eur",
        "area_m2",
        "price_per_m2",
        "habitaciones",
        "baños",
        "tags",
        "descripcion",
        "neighborhood_name_final",
        *global_special_flag_columns,
        "special_transaction_reasons",
        "Enlace",
    ],
].copy()

special_candidates_path = (
    INTERIM_DIR
    / "special_transaction_candidates.csv"
)

special_transaction_candidates.to_csv(
    special_candidates_path,
    index=False,
    encoding="utf-8-sig",
)

print("Candidatos guardados en:")
print(special_candidates_path)

Candidatos guardados en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\special_transaction_candidates.csv


In [139]:
# ============================================================
# Detección de posibles usos no residenciales
# ============================================================

# Es importante separar:
#
# 1. El tipo anunciado: piso, ático, dúplex, chalet...
# 2. El uso real descrito: hostal, hotel, oficina, local...
#
# Un anuncio puede comenzar por "Piso en venta", pero indicar
# en la descripción que actualmente funciona como hostal.


non_residential_candidate_patterns = {
    "hospitality": (
        r"\bhostal(?:es)?\b|"
        r"\bhotel(?:es)?\b|"
        r"\bpension(?:es)?\b|"
        r"\bapartamentos? turisticos?\b|"
        r"\bresidencia de estudiantes\b"
    ),

    "commercial_or_office": (
        r"\blocal comercial\b|"
        r"\buso comercial\b|"
        r"\buso como local\b|"
        r"\buso de oficina\b|"
        r"\buso como oficina\b|"
        r"\buso como despacho\b|"
        r"\bregistralmente local\b|"
        r"\bregistralmente oficina\b"
    ),

    "auxiliary_use": (
        r"\buso como trastero\b|"
        r"\bregistralmente trastero\b|"
        r"\buso como garaje\b|"
        r"\bregistralmente garaje\b"
    ),
}


non_residential_flag_columns = []

for category, pattern in (
    non_residential_candidate_patterns.items()
):
    flag_column = (
        f"flag_non_residential_{category}_candidate"
    )

    listings_cleaning[flag_column] = (
        listings_cleaning["combined_text_norm_all"]
        .str.contains(
            pattern,
            regex=True,
            na=False,
        )
    )

    non_residential_flag_columns.append(
        flag_column
    )


# Este indicador es deliberadamente amplio.
# Sirve para seleccionar anuncios que necesitan revisión,
# pero no implica que todos deban excluirse.

listings_cleaning[
    "flag_possible_non_residential_use"
] = (
    listings_cleaning[
        non_residential_flag_columns
    ].any(axis=1)
)


# Patrones de alta confianza.
# Aquí exigimos expresiones que indican directamente que
# el inmueble se vende o utiliza como negocio/local.

confirmed_non_residential_pattern = (
    r"\ba la venta (?:un |una )?"
    r"(?:hostal|hotel|pension|residencia de estudiantes)\b|"

    r"\bventa de (?:un |una )?"
    r"(?:hostal|hotel|pension|residencia de estudiantes)\b|"

    r"\b(?:hostal|hotel|pension) en venta\b|"

    r"\b(?:destinad[ao]|habilitad[ao]|explotad[ao]) "
    r"(?:actualmente )?(?:como|para) "
    r"(?:hostal|hotel|pension|apartamento turistico)\b|"

    r"\bfunciona (?:actualmente )?como "
    r"(?:hostal|hotel|pension|apartamento turistico)\b|"

    # Uso no residencial actual, no meramente posible.
    r"\bactualmente (?:tiene |con )?uso (?:como|de) "
    r"(?:hostal|hotel|pension|apartamento turistico|"
    r"local comercial|oficina|despacho)\b|"

    r"\buso actual (?:como|de) "
    r"(?:hostal|hotel|pension|apartamento turistico|"
    r"local comercial|oficina|despacho)\b|"

    r"\b(?:inmueble|piso|vivienda|propiedad|"
    r"fincas?(?: registral(?:es)?)?) con uso "
    r"(?:como|de) "
    r"(?:hostal|hotel|pension|apartamento turistico|"
    r"local comercial|oficina|despacho)\b|"

    r"\bregistralmente (?:es )?(?:un |una )?"
    r"(?:local|oficina|garaje|trastero)\b|"

    r"\bse trata de (?:un |una )?"
    r"(?:local comercial|oficina|hostal|hotel|pension)\b"
)


listings_cleaning[
    "flag_non_residential_use_high_confidence"
] = (
    listings_cleaning["combined_text_norm_all"]
    .str.contains(
        confirmed_non_residential_pattern,
        regex=True,
        na=False,
    )
)


# Combinamos:
#
# - los usos ya detectados en la revisión de valores extremos;
# - los usos no residenciales de alta confianza detectados
#   ahora en los 11.826 anuncios.

listings_cleaning[
    "flag_non_residential_use_confirmed"
] = (
    listings_cleaning[
        "flag_non_standard_use_reviewed"
    ]
    | listings_cleaning[
        "flag_non_residential_use_high_confidence"
    ]
)


def obtain_non_residential_reasons(row):
    reasons = []

    for category in non_residential_candidate_patterns:
        flag_column = (
            f"flag_non_residential_{category}_candidate"
        )

        if row[flag_column]:
            reasons.append(category)

    return "|".join(reasons)


listings_cleaning["non_residential_use_reasons"] = (
    listings_cleaning.apply(
        obtain_non_residential_reasons,
        axis=1,
    )
)


non_residential_candidates = listings_cleaning.loc[
    listings_cleaning[
        "flag_possible_non_residential_use"
    ],
    [
        "titulo",
        "property_type"
        if "property_type" in listings_cleaning.columns
        else "titulo",
        "price_eur",
        "area_m2",
        "habitaciones",
        "tags",
        "descripcion",
        "flag_possible_non_residential_use",
        "flag_non_residential_use_high_confidence",
        "flag_non_residential_use_confirmed",
        "non_residential_use_reasons",
        "Enlace",
    ],
].copy()


# Evitamos una columna repetida si property_type todavía
# no se ha creado en esta parte del notebook.

non_residential_candidates = (
    non_residential_candidates.loc[
        :,
        ~non_residential_candidates.columns.duplicated(),
    ]
)


non_residential_candidates_path = (
    INTERIM_DIR
    / "non_residential_use_candidates.csv"
)

non_residential_candidates.to_csv(
    non_residential_candidates_path,
    index=False,
    encoding="utf-8-sig",
)


print(
    "Posibles usos no residenciales para revisar:",
    listings_cleaning[
        "flag_possible_non_residential_use"
    ].sum(),
)

print(
    "Usos no residenciales de alta confianza:",
    listings_cleaning[
        "flag_non_residential_use_high_confidence"
    ].sum(),
)

print(
    "Usos no residenciales confirmados:",
    listings_cleaning[
        "flag_non_residential_use_confirmed"
    ].sum(),
)

print("Candidatos guardados en:")
print(non_residential_candidates_path)

Posibles usos no residenciales para revisar: 102
Usos no residenciales de alta confianza: 7
Usos no residenciales confirmados: 8
Candidatos guardados en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\non_residential_use_candidates.csv


In [140]:
# ============================================================
# Muestra para validación manual de operaciones especiales
# ============================================================

special_validation_samples = []

for category in special_transaction_patterns:
    flag_column = f"flag_special_{category}"

    category_candidates = listings_cleaning.loc[
        listings_cleaning[flag_column],
        [
            "titulo",
            "tags",
            "descripcion",
            "Enlace",
        ],
    ].copy()

    if len(category_candidates) == 0:
        continue

    sample_size = min(
        20,
        len(category_candidates),
    )

    category_sample = category_candidates.sample(
        n=sample_size,
        random_state=42,
    )

    category_sample[
        "categoria_revisada"
    ] = category

    special_validation_samples.append(
        category_sample
    )


if special_validation_samples:
    special_validation_sample = pd.concat(
        special_validation_samples,
        ignore_index=True,
    )

    special_validation_sample[
        "validacion_manual"
    ] = ""

    special_validation_sample[
        "observaciones_revision"
    ] = ""

else:
    special_validation_sample = pd.DataFrame()


special_validation_path = (
    INTERIM_DIR
    / "special_transaction_validation_sample.csv"
)

special_validation_sample.to_csv(
    special_validation_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Número de anuncios de la muestra:",
    len(special_validation_sample),
)

print("Muestra guardada en:")
print(special_validation_path)

Número de anuncios de la muestra: 87
Muestra guardada en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\special_transaction_validation_sample.csv


### Evaluación manual de las reglas de operaciones especiales

In [141]:
# ============================================================
# Evaluación manual de las reglas de operaciones especiales
# ============================================================

manual_validation_path = (
    INTERIM_DIR
    / "special_transaction_validation_reviewed.csv"
)

if manual_validation_path.exists():

    manual_validation = pd.read_csv(
        manual_validation_path
    )

    manual_validation["validacion_manual"] = (
        manual_validation["validacion_manual"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    allowed_values = {
        "correcta",
        "falso_positivo",
        "dudosa",
    }

    invalid_values = set(
        manual_validation["validacion_manual"].unique()
    ) - allowed_values

    if invalid_values:
        raise ValueError(
            "Existen valores no permitidos en "
            "'validacion_manual':\n"
            f"{invalid_values}"
        )

    validation_metrics = (
        manual_validation
        .groupby("categoria_revisada")
        .agg(
            sample_size=(
                "validacion_manual",
                "size",
            ),
            correct_detections=(
                "validacion_manual",
                lambda values: values.eq(
                    "correcta"
                ).sum(),
            ),
            false_positives=(
                "validacion_manual",
                lambda values: values.eq(
                    "falso_positivo"
                ).sum(),
            ),
            doubtful_cases=(
                "validacion_manual",
                lambda values: values.eq(
                    "dudosa"
                ).sum(),
            ),
        )
        .reset_index()
    )

    validation_metrics["evaluated_cases"] = (
        validation_metrics["correct_detections"]
        + validation_metrics["false_positives"]
    )

    validation_metrics["manual_precision"] = np.where(
        validation_metrics["evaluated_cases"].gt(0),
        (
            validation_metrics["correct_detections"]
            / validation_metrics["evaluated_cases"]
        ),
        np.nan,
    ).round(3)

    display(validation_metrics)

    evaluated_mask = (
        manual_validation["validacion_manual"]
        .isin(
            [
                "correcta",
                "falso_positivo",
            ]
        )
    )

    overall_precision = (
        manual_validation.loc[
            evaluated_mask,
            "validacion_manual",
        ]
        .eq("correcta")
        .mean()
    )

    print(
        "Precisión manual global:",
        round(overall_precision, 3),
    )

else:
    print("Validación manual pendiente.")
    print(
        "Completa la muestra y guárdala como:"
    )
    print(manual_validation_path)

,categoria_revisada,sample_size,correct_detections,false_positives,doubtful_cases,evaluated_cases,manual_precision
0,auction,20,20,0,0,20,1.00
1,bare_ownership_or_usufruct,20,20,0,0,20,1.00
2,occupied_or_no_possession,20,19,1,0,20,0.95
3,partial_ownership,7,7,0,0,7,1.00
4,tenant_or_rented,20,20,0,0,20,1.00


Precisión manual global: 0.989


In [142]:
# ============================================================
# Decisión final para el análisis de precio y superficie
# ============================================================

# Un anuncio puede presentar varias incidencias.
#
# El orden de prioridad es:
#
# 1. Error probable de superficie.
# 2. Uso no residencial confirmado.
# 3. Operación especial.
# 4. Mercado residencial convencional.

listings_cleaning["standard_market_status"] = np.select(
    [
        listings_cleaning[
            "flag_probable_area_error_reviewed"
        ],

        listings_cleaning[
            "flag_non_residential_use_confirmed"
        ],

        listings_cleaning[
            "flag_special_transaction_all"
        ],
    ],
    [
        "exclude_probable_area_error",
        "exclude_non_residential_use",
        "exclude_special_transaction",
    ],
    default="eligible_standard_market",
)


listings_cleaning["use_for_neighborhood_price"] = (
    listings_cleaning["standard_market_status"]
    == "eligible_standard_market"
)


display(
    listings_cleaning[
        "standard_market_status"
    ].value_counts()
)

standard_market_status
eligible_standard_market       11196
exclude_special_transaction      618
exclude_non_residential_use        8
exclude_probable_area_error        4
Name: count, dtype: int64

In [143]:
status_counts = (
    listings_cleaning[
        "standard_market_status"
    ]
    .value_counts()
)

print(
    "Total clasificado:",
    status_counts.sum(),
)

print(
    "Anuncios utilizables para calcular precios por barrio:",
    listings_cleaning[
        "use_for_neighborhood_price"
    ].sum(),
)

print(
    "Anuncios excluidos del mercado residencial estándar:",
    (
        ~listings_cleaning[
            "use_for_neighborhood_price"
        ]
    ).sum(),
)

print(
    "Anuncios sin estado:",
    listings_cleaning[
        "standard_market_status"
    ].isna().sum(),
)

Total clasificado: 11826
Anuncios utilizables para calcular precios por barrio: 11196
Anuncios excluidos del mercado residencial estándar: 630
Anuncios sin estado: 0


In [144]:
price_area_screened_path = (
    INTERIM_DIR
    / "listings_price_area_screened.csv"
)

listings_cleaning.to_csv(
    price_area_screened_path,
    index=False,
    encoding="utf-8-sig",
)

print("Dataset guardado en:")
print(price_area_screened_path)

Dataset guardado en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\listings_price_area_screened.csv


## Siguiente bloque: habitaciones y baños
### Convertir las variables a numéricas

In [145]:
for column in ["habitaciones", "baños"]:
    listings_cleaning[f"{column}_numeric"] = pd.to_numeric(
        listings_cleaning[column],
        errors="coerce",
    )

rooms_bathrooms_audit = pd.DataFrame(
    {
        "variable": ["habitaciones", "baños"],
        "nulos_originales": [
            listings_cleaning["habitaciones"].isna().sum(),
            listings_cleaning["baños"].isna().sum(),
        ],
        "nulos_despues_conversion": [
            listings_cleaning["habitaciones_numeric"].isna().sum(),
            listings_cleaning["baños_numeric"].isna().sum(),
        ],
        "ceros": [
            listings_cleaning["habitaciones_numeric"].eq(0).sum(),
            listings_cleaning["baños_numeric"].eq(0).sum(),
        ],
        "negativos": [
            listings_cleaning["habitaciones_numeric"].lt(0).sum(),
            listings_cleaning["baños_numeric"].lt(0).sum(),
        ],
    }
)

display(rooms_bathrooms_audit)

,variable,nulos_originales,nulos_despues_conversion,ceros,negativos
0,habitaciones,366,366,0,0
1,baños,0,0,9311,0


### Ver las distribuciones

In [146]:
print("Distribución de habitaciones:")
display(
    listings_cleaning[
        "habitaciones_numeric"
    ]
    .value_counts(dropna=False)
    .sort_index()
)

print("Distribución de baños:")
display(
    listings_cleaning[
        "baños_numeric"
    ]
    .value_counts(dropna=False)
    .sort_index()
)

Distribución de habitaciones:


habitaciones_numeric
1.0     1624
2.0     3383
3.0     3768
4.0     1490
5.0      690
6.0      291
7.0      102
8.0       48
9.0       26
10.0      15
11.0      10
12.0       2
13.0       3
15.0       3
16.0       3
17.0       1
20.0       1
NaN      366
Name: count, dtype: int64

Distribución de baños:


baños_numeric
0    9311
1    1063
2     989
3     319
4      87
5      34
6      17
7       6
Name: count, dtype: int64

### Comprobar si los ceros son realmente datos desconocidos

In [147]:
bathrooms_zero_sample = listings_cleaning.loc[
    listings_cleaning["baños_numeric"].eq(0),
    [
        "titulo",
        "habitaciones_numeric",
        "baños_numeric",
        "tags",
        "descripcion",
        "Enlace",
    ],
].sample(
    n=min(
        30,
        listings_cleaning["baños_numeric"].eq(0).sum(),
    ),
    random_state=42,
)

display(bathrooms_zero_sample)

,titulo,habitaciones_numeric,baños_numeric,tags,descripcion,Enlace
10982,"Piso en calle Mojácar, Apóstol Santiago, Madrid",2.0,0,NaN,NaN,https://www.idealista.com/inmueble/106789933/
159,"Piso en calle de Francisco Silvela, Lista, Madrid",2.0,0,"PISO,ARMARIOS,EXTERIOR",¡Venta de Piso en Calle de Francisco Silvela!Características del inmueble:Superficie: 80 m²2 habitaciones con armarios empotrados1 baño completoEs...,https://www.idealista.com/inmueble/106318036/
4438,"Piso en calle de Pinos Baja, Valdeacederas, Madrid",4.0,0,"VIVIENDA,PISO,TERRAZA,ESPECTACULAR,HOGAR","Si buscas una vivienda con encanto y en una ubicación cómoda, este piso en la calle Pinos Baja te va a encantar.Con sus 158m², este hogar es ideal...",https://www.idealista.com/inmueble/106426385/
5584,"Piso en Prosperidad, 18, Ciudad Jardín, Madrid",3.0,0,"PISO,HOGAR,NUEVO","¡Descubre tu nuevo hogar en Prosperidad, Chamartín!Este encantador piso de 3 habitaciones se encuentra en el corazón del barrio de Prosperidad, un...",https://www.idealista.com/inmueble/105336276/
3919,"Piso en Malasaña-Universidad, Madrid",3.0,0,"VIVIENDA,AMPLIO,VISTAS,EXCLUSIVA,REFORMADA,AMUEBLADA,METRO","Exclusiva vivienda reformada y amueblada al detalle, con calidades de primer nivel, situada en la cuarta planta de un edificio representativo, don...",https://www.idealista.com/inmueble/106690707/
10838,"Casa o chalet independiente en El Plantío, Madrid",8.0,0,"VIVIENDA,CASA,METRO","Beatriz Mateo-Sagasta de GV | ARAVACA comercializa la siguiente propiedad:Vivienda en Casaquemada, una de las mejores urbanizaciones de Madrid, ju...",https://www.idealista.com/inmueble/105313316/
5877,"Piso en Canillas, Madrid",4.0,0,"VIVIENDA,PISO,LUMINOSO,EXTERIOR","Luminoso piso exterior con gran potencial en CanillasEn el tranquilo y familiar barrio de Canillas, ponemos a la venta esta vivienda exterior en t...",https://www.idealista.com/inmueble/107245841/
5257,"Piso en calle de Nuestra Señora de Luján, Nueva España, Madrid",4.0,0,"VIVIENDA,TERRAZA,ÁTICO,CASA",Sacamos al mercado una de las casas más luminosas de todo el distrito de Chamartín sin que sea ático. Inicialmente la vivienda se distribuía en 4 ...,https://www.idealista.com/inmueble/107009797/
1693,"Piso en calle de Francisco Fatou, Casco Histórico de Vallecas, Madrid",2.0,0,"PISO,REFORMAR,PATIO","Piso con dos dormitorios, PATIO COMUNITARIO, salón, cocina y baño, PARA REFORMAR, con muchas posibilidades y muy bien comunicado. Mejor ver. Sin c...",https://www.idealista.com/inmueble/106676533/
3162,"Piso en calle Dodge, 4, Orcasitas, Madrid",2.0,0,"PISO,TERRAZA,AMPLIO,HOGAR,FUNCIONAL,NUEVO","¡Descubre tu nuevo hogar en el corazón de Orcasitas, Madrid! Este magnífico piso de 108 m² te ofrece un espacio acogedor y funcional, perfecto par...",https://www.idealista.com/inmueble/107158673/


### Buscar baños mencionados en el texto

In [148]:
# ============================================================
# Extracción del número de baños mencionado en el texto
# ============================================================

# El texto de combined_text_norm_all está normalizado:
# baño  -> bano
# baños -> banos

bathroom_number_words = {
    "un": 1,
    "uno": 1,
    "una": 1,
    "dos": 2,
    "tres": 3,
    "cuatro": 4,
    "cinco": 5,
    "seis": 6,
    "siete": 7,
    "ocho": 8,
    "nueve": 9,
    "diez": 10,
}

bathroom_pattern = re.compile(
    r"\b("
    r"\d{1,2}|"
    r"un|uno|una|dos|tres|cuatro|cinco|"
    r"seis|siete|ocho|nueve|diez"
    r")\s+"
    r"(?:cuartos?\s+de\s+)?"
    r"banos?\b"
)


def extract_bathroom_mentions(text):
    """
    Devuelve todos los números de baños mencionados
    explícitamente en el texto del anuncio.
    """
    if not isinstance(text, str) or not text:
        return []

    mentions = []

    for match in bathroom_pattern.finditer(text):
        value = match.group(1)

        if value.isdigit():
            number = int(value)
        else:
            number = bathroom_number_words[value]

        mentions.append(number)

    return mentions


listings_cleaning["bathroom_mentions_text"] = (
    listings_cleaning["combined_text_norm_all"]
    .apply(extract_bathroom_mentions)
)

listings_cleaning["bathroom_unique_mentions"] = (
    listings_cleaning["bathroom_mentions_text"]
    .apply(lambda values: sorted(set(values)))
)

listings_cleaning["bathrooms_inferred_text"] = (
    listings_cleaning["bathroom_unique_mentions"]
    .apply(
        lambda values: (
            values[0]
            if len(values) == 1
            else np.nan
        )
    )
)

listings_cleaning["bathrooms_text_status"] = np.select(
    [
        listings_cleaning[
            "bathroom_unique_mentions"
        ].apply(len).eq(0),

        listings_cleaning[
            "bathroom_unique_mentions"
        ].apply(len).eq(1),

        listings_cleaning[
            "bathroom_unique_mentions"
        ].apply(len).gt(1),
    ],
    [
        "not_found",
        "single_value",
        "ambiguous_multiple_values",
    ],
    default="unknown",
)

bathrooms_zero = listings_cleaning["baños_numeric"].eq(0)

print(
    "Anuncios con baños = 0:",
    bathrooms_zero.sum(),
)

print(
    "Baños = 0 y un único valor encontrado en el texto:",
    (
        bathrooms_zero
        & listings_cleaning[
            "bathrooms_text_status"
        ].eq("single_value")
    ).sum(),
)

print(
    "Baños = 0 y varios valores distintos en el texto:",
    (
        bathrooms_zero
        & listings_cleaning[
            "bathrooms_text_status"
        ].eq("ambiguous_multiple_values")
    ).sum(),
)

print(
    "Baños = 0 sin número localizado en el texto:",
    (
        bathrooms_zero
        & listings_cleaning[
            "bathrooms_text_status"
        ].eq("not_found")
    ).sum(),
)

display(
    listings_cleaning.loc[
        bathrooms_zero
        & listings_cleaning[
            "bathrooms_text_status"
        ].ne("not_found"),
        [
            "titulo",
            "baños_numeric",
            "bathroom_mentions_text",
            "bathroom_unique_mentions",
            "bathrooms_inferred_text",
            "bathrooms_text_status",
            "descripcion",
            "Enlace",
        ],
    ].head(30)
)

Anuncios con baños = 0: 9311
Baños = 0 y un único valor encontrado en el texto: 350
Baños = 0 y varios valores distintos en el texto: 5
Baños = 0 sin número localizado en el texto: 8956


,titulo,baños_numeric,bathroom_mentions_text,bathroom_unique_mentions,bathrooms_inferred_text,bathrooms_text_status,descripcion,Enlace
16,"Piso en calle de Sáhara, 46, Los Rosales, Madrid",0,[1],[1],1.0,single_value,"Fantástico piso en Villaverde en primera planta, consta de dos dormitorios, un cuarto de baño, salón comedor y cocina equipada, zona cercana de me...",https://www.idealista.com/inmueble/105872702/
26,"Piso en Buena Vista, Madrid",0,[1],[1],1.0,single_value,Te presentamos este estupendo piso en planta baja de 59 m² distribuidos de forma ideal para aprovechar al máximo cada rincón. Cuenta con 2 dormito...,https://www.idealista.com/inmueble/106991781/
37,"Piso en avenida de Niceto Alcalá Zamora, Sanchinarro, Madrid",0,[2],[2],2.0,single_value,"¡Descubre tu nuevo hogar en Sanchinarro, una de las zonas más exclusivas de Madrid! Este magnífico piso de 105 m² construidos y 90 m² útiles te of...",https://www.idealista.com/inmueble/107007331/
122,"Casa o chalet independiente en avenida Osa Mayor, 189, Aravaca, Madrid",0,[12],[12],12.0,single_value,"Fantástico unifamiliar, con excelentes vistas y amplia zona ajardinada con piscina. Nuevo a estrenar en Aravaca. Madrid Capital, con 1240 m2, 16 h...",https://www.idealista.com/inmueble/106030032/
199,"Dúplex en Valdemarín, Madrid",0,[2],[2],2.0,single_value,"AMBASSADOR Les ofrece este fantástico ático dúplex en la exclusiva zona de Valdemarín. Tiene 150 metros de terraza. Hall de entrada, aseo de visit...",https://www.idealista.com/inmueble/25473186/
212,"Ático en calle de la Paz, Sol, Madrid",0,[1],[1],1.0,single_value,"ZONA MADRID REAL ESTATE vende EN EXCLUSIVA piso de 49m2 para reformar, cuenta con TRES DORMITORIOS, baño, cocina y dos TRASTEROS, ubicado en el co...",https://www.idealista.com/inmueble/106205571/
215,"Piso en Goya, Madrid",0,[4],[4],4.0,single_value,"A4L Real Estate ofrece un estupendo piso en venta a 200m del retiro.216m2 construidos, 180m2 útiles, tiene 5 dormitorios y 4 cuartos de baño, ampl...",https://www.idealista.com/inmueble/101238924/
309,"Piso en calle de Jorge Juan, Recoletos, Madrid",0,[4],[4],4.0,single_value,"¡PISO 100% A ESTRENAR, CON REFORMA INTEGRAL, EN CALLE JORGE JUAN!Presentamos este ATRACTIVO piso RECIÉN REFORMADO, con 3 dormitorios y 4 cuartos d...",https://www.idealista.com/inmueble/99313626/
310,"Ático en calle de Fernán González, Ibiza, Madrid",0,[2],[2],2.0,single_value,"Villas de Bravante gestiona, la venta en exclusiva y sin cobrar honorarios al comprador, de este precioso ático todo exterior, con orientación Oes...",https://www.idealista.com/inmueble/106341333/
381,"Piso en calle de la Estrella, Malasaña-Universidad, Madrid",0,[1],[1],1.0,single_value,"Fantástico piso totalmente reformado y amueblado a escasos pasos de GRAN VÍA, PLAZA DE ESPAÑA y calle SAN BERNARDO.Excelente piso reformado en una...",https://www.idealista.com/inmueble/106567882/


### ### Validación de los baños extraídos del texto

In [149]:
# ============================================================
# Validación de los baños extraídos del texto
# ============================================================

bathrooms_missing_original = (
    listings_cleaning["baños_numeric"].isna()
    | listings_cleaning["baños_numeric"].eq(0)
)

bathrooms_single_value = (
    listings_cleaning["bathrooms_text_status"]
    .eq("single_value")
)

bathrooms_ambiguous = (
    listings_cleaning["bathrooms_text_status"]
    .eq("ambiguous_multiple_values")
)

bathrooms_not_found = (
    listings_cleaning["bathrooms_text_status"]
    .eq("not_found")
)

bathrooms_recoverable = (
    bathrooms_missing_original
    & bathrooms_single_value
)

print(
    "Baños ausentes o iguales a 0 en el dato original:",
    bathrooms_missing_original.sum(),
)

print(
    "Recuperables con un único valor en el texto:",
    bathrooms_recoverable.sum(),
)

print(
    "Con varios valores distintos en el texto:",
    (
        bathrooms_missing_original
        & bathrooms_ambiguous
    ).sum(),
)

print(
    "Sin ningún número de baños localizado:",
    (
        bathrooms_missing_original
        & bathrooms_not_found
    ).sum(),
)

print(
    "Porcentaje recuperable:",
    round(
        bathrooms_recoverable.sum()
        / bathrooms_missing_original.sum()
        * 100,
        2,
    ),
    "%",
)

Baños ausentes o iguales a 0 en el dato original: 9311
Recuperables con un único valor en el texto: 350
Con varios valores distintos en el texto: 5
Sin ningún número de baños localizado: 8956
Porcentaje recuperable: 3.76 %


In [150]:
# ============================================================
# Distribución de los valores recuperables
# ============================================================

bathrooms_inferred_distribution = (
    listings_cleaning.loc[
        bathrooms_recoverable,
        "bathrooms_inferred_text",
    ]
    .value_counts()
    .sort_index()
    .rename_axis("numero_baños")
    .reset_index(name="numero_anuncios")
)

display(bathrooms_inferred_distribution)

,numero_baños,numero_anuncios
0,1.0,159
1,2.0,122
2,3.0,45
3,4.0,15
4,5.0,5
5,6.0,1
6,7.0,1
7,12.0,1
8,15.0,1


In [151]:
# ============================================================
# Muestra aleatoria para validar manualmente la extracción
# ============================================================

bathrooms_validation_sample = listings_cleaning.loc[
    bathrooms_recoverable,
    [
        "titulo",
        "habitaciones_numeric",
        "baños_numeric",
        "bathrooms_inferred_text",
        "bathroom_unique_mentions",
        "descripcion",
        "Enlace",
    ],
].sample(
    n=min(
        40,
        bathrooms_recoverable.sum(),
    ),
    random_state=42,
)

display(bathrooms_validation_sample)

,titulo,habitaciones_numeric,baños_numeric,bathrooms_inferred_text,bathroom_unique_mentions,descripcion,Enlace
5172,"Piso en calle de Artesa de Segre, Ciudad Universitaria, Madrid",1.0,0,1.0,[1],"Laura Pérez-Hernández de GV | PUERTA DE HIERRO comercializa la siguiente vivienda:Piso de 80 metros cuadrados construidos, situado en urbanización...",https://www.idealista.com/inmueble/105289238/
11340,"Piso en Goya, Madrid",4.0,0,1.0,[1],"Fantástico piso señorial, de 323 m2 construidos (280 m2 útiles). Prácticamente todo exterior (excepto la cocina, el aseo y un baño), orientado a E...",https://www.idealista.com/inmueble/106547352/
10180,"Piso en Villaverde Alto, Madrid",2.0,0,2.0,[2],¡Entra en el estilo de vida moderno con este impresionante loft en venta! Este inmueble de dos plantas cuenta con dos habitaciones y dos cuartos d...,https://www.idealista.com/inmueble/104676024/
7835,"Piso en Chueca-Justicia, Madrid",3.0,0,2.0,[2],"Bonito piso exterior con tres grandes ventanales a la calle, en edificio muy cuidado de 1900, en ubicación excepcional, a escasos metros de la Pla...",https://www.idealista.com/inmueble/106269784/
5121,"Piso en calle Puebla de Sanabria, Villaverde Alto, Madrid",3.0,0,1.0,[1],PISO OCUPADO POR PERSONA SIN JUSTO TÍTULOESTE INMUEBLE DEBIDO A SU ESTADO OCUPACIONAL NO ADMITE VISITASIDEAL PARA INVERSORESAmplia vivienda de 98 ...,https://www.idealista.com/inmueble/105540166/
8960,"Piso en Lista, Madrid",2.0,0,2.0,[2],INFOCASA pone a la venta una vivienda en el pleno Barrio Salamanca.El barrio de Salamanca es uno de los más conocidos y distinguidos de Madrid.El ...,https://www.idealista.com/inmueble/103419287/
9819,"Piso en calle Tuya, 12 de Octubre-Orcasur, Madrid",3.0,0,2.0,[2],"LINKCASA PASEO EXTREMADURA VENDE un piso de 115 m² ideal para INVERSIÓN en la calle Tuya, muy cerca de la Avenida Andalucía. La vivienda consta de...",https://www.idealista.com/inmueble/106663494/
7687,"Piso en paseo de las Delicias, Delicias, Madrid",1.0,0,1.0,[1],"“JOWNER VENDE VIVIENDA EN PASEO DE LAS DELICIAS”JOWNER vende piso exterior situado en una primera planta, en la emblemática zona del Paseo de las ...",https://www.idealista.com/inmueble/106775213/
9021,"Piso en calle de Benimamet, San Cristóbal, Madrid",3.0,0,1.0,[1],"INMOBILIARIA LÓPEZ & MORENO vende fabuloso piso, totalmente reformado, exterior de 60m2, se distribuye en 2 dormitorios (antes 3), salón-comedor, ...",https://www.idealista.com/inmueble/107024171/
6227,"Piso en calle de Goya, Goya, Madrid",2.0,0,1.0,[1],"Precioso piso en una primera planta exterior con tres balcones, muy luminoso. Cuenta con salón comedor, cocina independiente amueblada y equipada,...",https://www.idealista.com/inmueble/106461136/


### Comprobar valores elevados

In [152]:
# ============================================================
# Revisión de valores de baños elevados
# ============================================================

high_bathrooms_mask = (
    bathrooms_recoverable
    & listings_cleaning["bathrooms_inferred_text"].gt(10)
)

print(
    "Anuncios recuperables con más de 10 baños:",
    high_bathrooms_mask.sum(),
)

display(
    listings_cleaning.loc[
        high_bathrooms_mask,
        [
            "titulo",
            "area_m2",
            "habitaciones_numeric",
            "bathrooms_inferred_text",
            "descripcion",
            "Enlace",
        ],
    ]
)

Anuncios recuperables con más de 10 baños: 2


,titulo,area_m2,habitaciones_numeric,bathrooms_inferred_text,descripcion,Enlace
122,"Casa o chalet independiente en avenida Osa Mayor, 189, Aravaca, Madrid",1240,16.0,12.0,"Fantástico unifamiliar, con excelentes vistas y amplia zona ajardinada con piscina. Nuevo a estrenar en Aravaca. Madrid Capital, con 1240 m2, 16 h...",https://www.idealista.com/inmueble/106030032/
7713,"Piso en Palacio, Madrid",470,15.0,15.0,"¡ARTECASAS PONE A LA VENTA HOSTAL EN EL CENTRO DE MADRID!*DISPONEMOS DE HOSTALES*Hostal recién reformado de más de 450m2, que consta de 15 habitac...",https://www.idealista.com/inmueble/107126513/


### Construcción de las variables limpias de habitaciones y baños

In [153]:
# ------------------------------------------------------------
# Habitaciones
# ------------------------------------------------------------

# Conservamos el valor original cuando es válido.
# Los valores ausentes permanecen como NaN:
# no se rellenan mediante media, mediana ni moda.

listings_cleaning["rooms_clean"] = (
    listings_cleaning["habitaciones_numeric"]
    .where(
        listings_cleaning["habitaciones_numeric"].gt(0),
        np.nan,
    )
)

listings_cleaning["rooms_source"] = np.where(
    listings_cleaning["rooms_clean"].notna(),
    "original",
    "missing",
)


# ------------------------------------------------------------
# Baños
# ------------------------------------------------------------

# Los ceros se interpretan como ausencia de información,
# no como viviendas sin baño.

listings_cleaning["bathrooms_clean"] = (
    listings_cleaning["baños_numeric"]
    .where(
        listings_cleaning["baños_numeric"].gt(0),
        np.nan,
    )
)

# Solo recuperamos el valor desde el texto cuando:
# 1. El dato original es ausente o igual a cero.
# 2. En el texto aparece un único número de baños.

bathrooms_recovered_from_text = (
    bathrooms_missing_original
    & bathrooms_single_value
)

listings_cleaning.loc[
    bathrooms_recovered_from_text,
    "bathrooms_clean",
] = listings_cleaning.loc[
    bathrooms_recovered_from_text,
    "bathrooms_inferred_text",
]

# Registramos el origen de cada valor.
listings_cleaning["bathrooms_source"] = np.select(
    [
        listings_cleaning["baños_numeric"].gt(0),
        bathrooms_recovered_from_text,
        bathrooms_missing_original & bathrooms_ambiguous,
        bathrooms_missing_original & bathrooms_not_found,
    ],
    [
        "original",
        "recovered_from_text",
        "ambiguous_text",
        "missing",
    ],
    default="review",
)

# Indicadores de trazabilidad.
listings_cleaning["flag_bathrooms_recovered_text"] = (
    listings_cleaning["bathrooms_source"]
    .eq("recovered_from_text")
)

listings_cleaning["flag_bathrooms_ambiguous"] = (
    listings_cleaning["bathrooms_source"]
    .eq("ambiguous_text")
)

# Los valores elevados no se eliminan automáticamente.
# Se conservan marcados para su posterior revisión junto
# con el tipo de inmueble.
listings_cleaning["flag_bathrooms_high_review"] = (
    listings_cleaning["bathrooms_clean"].gt(10)
)


# ------------------------------------------------------------
# Validación
# ------------------------------------------------------------

print("Habitaciones disponibles:")
print(listings_cleaning["rooms_clean"].notna().sum())

print("Habitaciones desconocidas:")
print(listings_cleaning["rooms_clean"].isna().sum())

print("\nOrigen de los valores de baños:")
display(
    listings_cleaning["bathrooms_source"]
    .value_counts(dropna=False)
)

print(
    "Baños disponibles después de la recuperación:",
    listings_cleaning["bathrooms_clean"].notna().sum(),
)

print(
    "Baños todavía desconocidos o ambiguos:",
    listings_cleaning["bathrooms_clean"].isna().sum(),
)

print(
    "Valores de baños superiores a 10:",
    listings_cleaning[
        "flag_bathrooms_high_review"
    ].sum(),
)

Habitaciones disponibles:
11460
Habitaciones desconocidas:
366

Origen de los valores de baños:


bathrooms_source
missing                8956
original               2515
recovered_from_text     350
ambiguous_text            5
Name: count, dtype: int64

Baños disponibles después de la recuperación: 2865
Baños todavía desconocidos o ambiguos: 8961
Valores de baños superiores a 10: 2


### Limpieza de ascensor y localización interior/exterior

In [154]:
# ============================================================
# Limpieza de ascensor
# ============================================================

listings_cleaning["elevator_original_norm"] = (
    listings_cleaning["ascensor"]
    .apply(normalizar_texto_reglas)
)

elevator_mapping = {
    "s": "yes",
    "si": "yes",
    "con ascensor": "yes",
    "true": "yes",
    "1": "yes",
    "n": "no",
    "no": "no",
    "sin ascensor": "no",
    "false": "no",
    "0": "no",
}

listings_cleaning["elevator_status"] = (
    listings_cleaning["elevator_original_norm"]
    .map(elevator_mapping)
    .fillna("unknown")
)

listings_cleaning["elevator_source"] = np.where(
    listings_cleaning["elevator_status"].ne("unknown"),
    "original",
    "missing",
)

# Recuperación conservadora desde el texto.
text_elevator_yes = (
    listings_cleaning["combined_text_norm_all"]
    .str.contains(
        (
            r"\bcon ascensor\b|"
            r"\bdispone de ascensor\b|"
            r"\bfinca con ascensor\b|"
            r"\bedificio con ascensor\b"
        ),
        regex=True,
        na=False,
    )
)

text_elevator_no = (
    listings_cleaning["combined_text_norm_all"]
    .str.contains(
        (
            r"\bsin ascensor\b|"
            r"\bno tiene ascensor\b|"
            r"\bno dispone de ascensor\b|"
            r"\bfinca sin ascensor\b"
        ),
        regex=True,
        na=False,
    )
)

unknown_elevator = (
    listings_cleaning["elevator_status"]
    .eq("unknown")
)

recover_elevator_yes = (
    unknown_elevator
    & text_elevator_yes
    & ~text_elevator_no
)

recover_elevator_no = (
    unknown_elevator
    & text_elevator_no
    & ~text_elevator_yes
)

listings_cleaning.loc[
    recover_elevator_yes,
    "elevator_status",
] = "yes"

listings_cleaning.loc[
    recover_elevator_no,
    "elevator_status",
] = "no"

listings_cleaning.loc[
    recover_elevator_yes | recover_elevator_no,
    "elevator_source",
] = "recovered_from_text"

listings_cleaning["elevator_binary"] = (
    listings_cleaning["elevator_status"]
    .map(
        {
            "yes": 1,
            "no": 0,
        }
    )
    .astype("Float64")
)


# ============================================================
# Limpieza de localización interior/exterior
# ============================================================

listings_cleaning["location_original_norm"] = (
    listings_cleaning["localizacion"]
    .apply(normalizar_texto_reglas)
)

location_mapping = {
    "exterior": "exterior",
    "e": "exterior",
    "interior": "interior",
    "i": "interior",
}

listings_cleaning["location_status"] = (
    listings_cleaning["location_original_norm"]
    .map(location_mapping)
    .fillna("unknown")
)

listings_cleaning["location_source"] = np.where(
    listings_cleaning["location_status"].ne("unknown"),
    "original",
    "missing",
)

# Para evitar falsos positivos, buscamos únicamente
# en el título y las etiquetas, no en toda la descripción.

listings_cleaning["short_text_norm"] = (
    listings_cleaning["titulo"].fillna("")
    + " "
    + listings_cleaning["tags"].fillna("")
).apply(normalizar_texto_reglas)

text_exterior = (
    listings_cleaning["short_text_norm"]
    .str.contains(
        r"\bexterior\b",
        regex=True,
        na=False,
    )
)

text_interior = (
    listings_cleaning["short_text_norm"]
    .str.contains(
        r"\binterior\b",
        regex=True,
        na=False,
    )
)

unknown_location = (
    listings_cleaning["location_status"]
    .eq("unknown")
)

recover_exterior = (
    unknown_location
    & text_exterior
    & ~text_interior
)

recover_interior = (
    unknown_location
    & text_interior
    & ~text_exterior
)

listings_cleaning.loc[
    recover_exterior,
    "location_status",
] = "exterior"

listings_cleaning.loc[
    recover_interior,
    "location_status",
] = "interior"

listings_cleaning.loc[
    recover_exterior | recover_interior,
    "location_source",
] = "recovered_from_text"

listings_cleaning["exterior_binary"] = (
    listings_cleaning["location_status"]
    .map(
        {
            "exterior": 1,
            "interior": 0,
        }
    )
    .astype("Float64")
)

print("Origen del ascensor:")
display(
    listings_cleaning["elevator_source"]
    .value_counts(dropna=False)
)

print("Estado final del ascensor:")
display(
    listings_cleaning["elevator_status"]
    .value_counts(dropna=False)
)

print("Origen de interior/exterior:")
display(
    listings_cleaning["location_source"]
    .value_counts(dropna=False)
)

print("Estado final de interior/exterior:")
display(
    listings_cleaning["location_status"]
    .value_counts(dropna=False)
)

Origen del ascensor:


elevator_source
original               11033
missing                  772
recovered_from_text       21
Name: count, dtype: int64

Estado final del ascensor:


elevator_status
yes        8438
no         2616
unknown     772
Name: count, dtype: int64

Origen de interior/exterior:


location_source
original               10730
missing                  992
recovered_from_text      104
Name: count, dtype: int64

Estado final de interior/exterior:


location_status
exterior    9133
interior    1701
unknown      992
Name: count, dtype: int64

### Limpieza y normalización de la planta

In [155]:
# ============================================================
# Limpieza de planta
# ============================================================

listings_cleaning["floor_original_norm"] = (
    listings_cleaning["planta"]
    .apply(normalizar_texto_reglas)
)

listings_cleaning["floor_number"] = pd.to_numeric(
    listings_cleaning["planta"],
    errors="coerce",
)

listings_cleaning["floor_category"] = np.where(
    listings_cleaning["floor_number"].notna(),
    "numeric",
    "unknown",
)

floor_text = listings_cleaning["floor_original_norm"]

basement_mask = floor_text.str.contains(
    r"\bsotano\b",
    regex=True,
    na=False,
)

semi_basement_mask = floor_text.str.contains(
    r"\bsemisotano\b",
    regex=True,
    na=False,
)

ground_floor_mask = floor_text.str.contains(
    r"\bbajo\b|\bplanta baja\b",
    regex=True,
    na=False,
)

mezzanine_mask = floor_text.str.contains(
    r"\bentreplanta\b",
    regex=True,
    na=False,
)

principal_mask = floor_text.str.contains(
    r"\bprincipal\b",
    regex=True,
    na=False,
)

listings_cleaning.loc[
    basement_mask,
    ["floor_number", "floor_category"],
] = [-1, "basement"]

listings_cleaning.loc[
    semi_basement_mask,
    ["floor_number", "floor_category"],
] = [-0.5, "semi_basement"]

listings_cleaning.loc[
    ground_floor_mask,
    ["floor_number", "floor_category"],
] = [0, "ground_floor"]

listings_cleaning.loc[
    mezzanine_mask,
    ["floor_number", "floor_category"],
] = [0.5, "mezzanine"]

# "Principal" se conserva como categoría,
# pero no se convierte en número arbitrariamente.
listings_cleaning.loc[
    principal_mask,
    "floor_category",
] = "principal"

# Recuperamos números escritos en formatos como "4ª" o "planta 4".
numeric_floor_candidate = (
    floor_text
    .str.extract(
        r"(-?\d+(?:[.,]\d+)?)",
        expand=False,
    )
    .str.replace(",", ".", regex=False)
)

numeric_floor_candidate = pd.to_numeric(
    numeric_floor_candidate,
    errors="coerce",
)

recover_numeric_floor = (
    listings_cleaning["floor_category"].eq("unknown")
    & numeric_floor_candidate.notna()
)

listings_cleaning.loc[
    recover_numeric_floor,
    "floor_number",
] = numeric_floor_candidate.loc[
    recover_numeric_floor
]

listings_cleaning.loc[
    recover_numeric_floor,
    "floor_category",
] = "numeric_from_text"

listings_cleaning["floor_source"] = np.select(
    [
        listings_cleaning["floor_category"].eq("numeric"),
        listings_cleaning["floor_category"].isin(
            [
                "basement",
                "semi_basement",
                "ground_floor",
                "mezzanine",
                "principal",
            ]
        ),
        listings_cleaning["floor_category"].eq(
            "numeric_from_text"
        ),
    ],
    [
        "original_numeric",
        "original_category",
        "parsed_from_original_text",
    ],
    default="missing",
)

# Las plantas originalmente numéricas con valor negativo
# representan niveles de sótano.

negative_original_floor_mask = (
    pd.to_numeric(
        listings_cleaning["planta"],
        errors="coerce",
    ).lt(0)
    & listings_cleaning["floor_category"].eq("numeric")
)

listings_cleaning.loc[
    negative_original_floor_mask,
    "floor_category",
] = "basement"

listings_cleaning.loc[
    negative_original_floor_mask,
    "floor_source",
] = "original_numeric"

print("Categorías de planta:")
display(
    listings_cleaning["floor_category"]
    .value_counts(dropna=False)
)

print("Origen de la planta:")
display(
    listings_cleaning["floor_source"]
    .value_counts(dropna=False)
)

Categorías de planta:


floor_category
numeric_from_text    8780
ground_floor         1509
unknown              1225
mezzanine             163
basement              149
Name: count, dtype: int64

Origen de la planta:


floor_source
parsed_from_original_text    8780
original_category            1800
missing                      1225
original_numeric               21
Name: count, dtype: int64

### Extracción del tipo de inmueble

In [156]:
# ============================================================
# Extracción del tipo de inmueble desde la etiqueta inicial
# ============================================================

# El tipo de inmueble aparece al comienzo del título,
# antes de la palabra "en".

listings_cleaning["property_label_original"] = (
    listings_cleaning["titulo"]
    .str.extract(
        r"^\s*(.*?)\s+en\b",
        expand=False,
    )
)

listings_cleaning["property_label_norm"] = (
    listings_cleaning["property_label_original"]
    .apply(normalizar_texto_reglas)
)

property_type_mapping = {
    "piso": "flat",
    "atico": "penthouse",
    "duplex": "duplex",
    "estudio": "studio",
    "casa o chalet independiente": "house_or_chalet",
    "chalet adosado": "house_or_chalet",
    "chalet pareado": "house_or_chalet",
    "chalet": "house_or_chalet",
    "casa rural": "house_or_chalet",
}

listings_cleaning["property_type"] = (
    listings_cleaning["property_label_norm"]
    .map(property_type_mapping)
)

# Si aparece un tipo nuevo no contemplado,
# detenemos el notebook para revisarlo.

if listings_cleaning["property_type"].isna().any():
    unknown_labels = (
        listings_cleaning.loc[
            listings_cleaning["property_type"].isna(),
            "property_label_original",
        ]
        .value_counts(dropna=False)
    )

    raise ValueError(
        "Existen tipos de inmueble sin clasificar:\n"
        f"{unknown_labels}"
    )

listings_cleaning["property_type_source"] = (
    "initial_title_label"
)

# Las etiquetas iniciales del dataset son residenciales.
# El posible uso real no residencial se analiza aparte
# mediante el título, las etiquetas y la descripción.

listings_cleaning[
    "flag_non_residential_property_type"
] = False

print("Tipos de inmueble:")

display(
    listings_cleaning["property_type"]
    .value_counts(dropna=False)
)

print(
    "Tipologías no residenciales según la etiqueta inicial:",
    listings_cleaning[
        "flag_non_residential_property_type"
    ].sum(),
)

Tipos de inmueble:


property_type
flat               9551
penthouse           782
house_or_chalet     719
duplex              435
studio              339
Name: count, dtype: int64

Tipologías no residenciales según la etiqueta inicial: 0


### Auditoría conjunta y guardado

In [157]:
# ============================================================
# Auditoría conjunta de atributos inmobiliarios
# ============================================================

attribute_quality_summary = pd.DataFrame(
    {
        "indicador": [
            "total_anuncios",
            "habitaciones_desconocidas",
            "baños_desconocidos_o_ambiguos",
            "ascensor_desconocido",
            "interior_exterior_desconocido",
            "planta_desconocida",
            "tipo_inmueble_sin_clasificar",
            "usos_no_residenciales_confirmados",
            "utilizables_precio_barrio",
        ],
        "numero_anuncios": [
            len(listings_cleaning),

            listings_cleaning[
                "rooms_clean"
            ].isna().sum(),

            listings_cleaning[
                "bathrooms_clean"
            ].isna().sum(),

            listings_cleaning[
                "elevator_status"
            ].eq("unknown").sum(),

            listings_cleaning[
                "location_status"
            ].eq("unknown").sum(),

            listings_cleaning[
                "floor_category"
            ].eq("unknown").sum(),

            listings_cleaning[
                "property_type"
            ].isna().sum(),

            listings_cleaning[
                "flag_non_residential_use_confirmed"
            ].sum(),

            listings_cleaning[
                "use_for_neighborhood_price"
            ].sum(),
        ],
    }
)

attribute_quality_summary["porcentaje"] = (
    attribute_quality_summary["numero_anuncios"]
    / len(listings_cleaning)
    * 100
).round(2)

display(attribute_quality_summary)

print(
    "Enlaces duplicados:",
    listings_cleaning[
        "Enlace"
    ].duplicated().sum(),
)

print(
    "Total final en esta etapa:",
    len(listings_cleaning),
)

attributes_cleaned_path = (
    INTERIM_DIR
    / "listings_attributes_cleaned.csv"
)

listings_cleaning.to_csv(
    attributes_cleaned_path,
    index=False,
    encoding="utf-8-sig",
)

print("Dataset guardado en:")
print(attributes_cleaned_path)

,indicador,numero_anuncios,porcentaje
0,total_anuncios,11826,100.00
1,habitaciones_desconocidas,366,3.09
2,baños_desconocidos_o_ambiguos,8961,75.77
3,ascensor_desconocido,772,6.53
4,interior_exterior_desconocido,992,8.39
5,planta_desconocida,1225,10.36
6,tipo_inmueble_sin_clasificar,0,0.00
7,usos_no_residenciales_confirmados,8,0.07
8,utilizables_precio_barrio,11196,94.67


Enlaces duplicados: 0
Total final en esta etapa: 11826
Dataset guardado en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\listings_attributes_cleaned.csv


In [158]:
import hashlib


def normalizar_texto_duplicados(valor):
    """
    Normalización más estricta para comparar anuncios:
    elimina tildes, puntuación y espacios repetidos.
    """
    texto = normalizar_texto_reglas(valor)

    texto = re.sub(
        r"[^a-z0-9]+",
        " ",
        texto,
    )

    return re.sub(
        r"\s+",
        " ",
        texto,
    ).strip()


listings_cleaning["duplicate_title_norm"] = (
    listings_cleaning["titulo"]
    .apply(normalizar_texto_duplicados)
)

listings_cleaning["duplicate_description_norm"] = (
    listings_cleaning["descripcion"]
    .apply(normalizar_texto_duplicados)
)

# Solo aplicamos esta regla estricta cuando existe
# una descripción suficientemente informativa.
listings_cleaning["duplicate_check_eligible"] = (
    listings_cleaning[
        "duplicate_description_norm"
    ].str.len().ge(80)
)

fingerprint_columns = pd.DataFrame(
    {
        "zone_key": (
            listings_cleaning["zone_key_final"]
            .astype("string")
            .fillna("<missing>")
        ),
        "title": (
            listings_cleaning["duplicate_title_norm"]
        ),
        "price": (
            listings_cleaning["price_eur"]
            .astype("string")
            .fillna("<missing>")
        ),
        "area": (
            listings_cleaning["area_m2"]
            .astype("string")
            .fillna("<missing>")
        ),
        "rooms": (
            listings_cleaning["rooms_clean"]
            .astype("string")
            .fillna("<missing>")
        ),
        "bathrooms": (
            listings_cleaning["bathrooms_clean"]
            .astype("string")
            .fillna("<missing>")
        ),
        "description": (
            listings_cleaning[
                "duplicate_description_norm"
            ]
        ),
    }
)

fingerprint_text = (
    fingerprint_columns
    .astype(str)
    .agg("||".join, axis=1)
)

listings_cleaning["duplicate_fingerprint"] = (
    fingerprint_text.apply(
        lambda value: hashlib.sha256(
            value.encode("utf-8")
        ).hexdigest()
    )
)

eligible_fingerprints = listings_cleaning.loc[
    listings_cleaning["duplicate_check_eligible"],
    "duplicate_fingerprint",
]

fingerprint_counts = (
    eligible_fingerprints.value_counts()
)

listings_cleaning["duplicate_group_size"] = (
    listings_cleaning["duplicate_fingerprint"]
    .map(fingerprint_counts)
    .fillna(1)
    .astype(int)
)

listings_cleaning["flag_strict_duplicate_candidate"] = (
    listings_cleaning["duplicate_check_eligible"]
    & listings_cleaning["duplicate_group_size"].gt(1)
)

candidate_fingerprints = sorted(
    listings_cleaning.loc[
        listings_cleaning[
            "flag_strict_duplicate_candidate"
        ],
        "duplicate_fingerprint",
    ].unique()
)

fingerprint_group_mapping = {
    fingerprint: group_number
    for group_number, fingerprint
    in enumerate(candidate_fingerprints, start=1)
}

listings_cleaning["duplicate_group_id"] = (
    listings_cleaning["duplicate_fingerprint"]
    .map(fingerprint_group_mapping)
    .astype("Int64")
)

print(
    "Anuncios candidatos a duplicado estricto:",
    listings_cleaning[
        "flag_strict_duplicate_candidate"
    ].sum(),
)

print(
    "Grupos de posibles duplicados:",
    listings_cleaning.loc[
        listings_cleaning[
            "flag_strict_duplicate_candidate"
        ],
        "duplicate_group_id",
    ].nunique(),
)

print("Distribución por tamaño de grupo:")
display(
    listings_cleaning.loc[
        listings_cleaning[
            "flag_strict_duplicate_candidate"
        ],
        "duplicate_group_size",
    ].value_counts().sort_index()
)

Anuncios candidatos a duplicado estricto: 226
Grupos de posibles duplicados: 107
Distribución por tamaño de grupo:


duplicate_group_size
2    192
3     30
4      4
Name: count, dtype: int64

In [159]:
duplicate_candidates = listings_cleaning.loc[
    listings_cleaning[
        "flag_strict_duplicate_candidate"
    ],
    [
        "duplicate_group_id",
        "duplicate_group_size",
        "titulo",
        "price_eur",
        "area_m2",
        "rooms_clean",
        "bathrooms_clean",
        "property_type",
        "neighborhood_name_final",
        "descripcion",
        "Enlace",
    ],
].copy()

duplicate_candidates = duplicate_candidates.sort_values(
    [
        "duplicate_group_id",
        "Enlace",
    ]
)

duplicate_candidates_path = (
    INTERIM_DIR
    / "duplicate_candidates_strict.csv"
)

duplicate_candidates.to_csv(
    duplicate_candidates_path,
    index=False,
    encoding="utf-8-sig",
)

duplicate_screened_path = (
    INTERIM_DIR
    / "listings_duplicate_screened.csv"
)

listings_cleaning.to_csv(
    duplicate_screened_path,
    index=False,
    encoding="utf-8-sig",
)

print("Candidatos guardados en:")
print(duplicate_candidates_path)

print("Dataset completo guardado en:")
print(duplicate_screened_path)

Candidatos guardados en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\duplicate_candidates_strict.csv
Dataset completo guardado en:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\interim\listings_duplicate_screened.csv


### Refinamiento de la detección de duplicados

In [160]:
# ============================================================

# ============================================================

# Incorporamos planta, ascensor y condición interior/exterior.
# Dos anuncios solo serán candidatos estrictos si coinciden
# también en estas características.

refined_fingerprint_columns = pd.DataFrame(
    {
        "zone_key": (
            listings_cleaning["zone_key_final"]
            .astype("string")
            .fillna("<missing>")
        ),
        "title": listings_cleaning["duplicate_title_norm"],
        "price": (
            listings_cleaning["price_eur"]
            .astype("string")
            .fillna("<missing>")
        ),
        "area": (
            listings_cleaning["area_m2"]
            .astype("string")
            .fillna("<missing>")
        ),
        "rooms": (
            listings_cleaning["rooms_clean"]
            .astype("string")
            .fillna("<missing>")
        ),
        "bathrooms": (
            listings_cleaning["bathrooms_clean"]
            .astype("string")
            .fillna("<missing>")
        ),
        "property_type": (
            listings_cleaning["property_type"]
            .astype("string")
            .fillna("<missing>")
        ),
        "elevator": (
            listings_cleaning["elevator_status"]
            .astype("string")
            .fillna("<missing>")
        ),
        "location": (
            listings_cleaning["location_status"]
            .astype("string")
            .fillna("<missing>")
        ),
        "floor_number": (
            listings_cleaning["floor_number"]
            .astype("string")
            .fillna("<missing>")
        ),
        "floor_category": (
            listings_cleaning["floor_category"]
            .astype("string")
            .fillna("<missing>")
        ),
        "description": (
            listings_cleaning["duplicate_description_norm"]
        ),
    }
)

refined_fingerprint_text = (
    refined_fingerprint_columns
    .astype(str)
    .agg("||".join, axis=1)
)

listings_cleaning["duplicate_fingerprint_refined"] = (
    refined_fingerprint_text.apply(
        lambda value: hashlib.sha256(
            value.encode("utf-8")
        ).hexdigest()
    )
)

eligible_refined_fingerprints = listings_cleaning.loc[
    listings_cleaning["duplicate_check_eligible"],
    "duplicate_fingerprint_refined",
]

refined_fingerprint_counts = (
    eligible_refined_fingerprints.value_counts()
)

listings_cleaning["duplicate_group_size_refined"] = (
    listings_cleaning["duplicate_fingerprint_refined"]
    .map(refined_fingerprint_counts)
    .fillna(1)
    .astype(int)
)

listings_cleaning["flag_duplicate_refined_candidate"] = (
    listings_cleaning["duplicate_check_eligible"]
    & listings_cleaning[
        "duplicate_group_size_refined"
    ].gt(1)
)

refined_fingerprints = sorted(
    listings_cleaning.loc[
        listings_cleaning[
            "flag_duplicate_refined_candidate"
        ],
        "duplicate_fingerprint_refined",
    ].unique()
)

refined_group_mapping = {
    fingerprint: group_number
    for group_number, fingerprint
    in enumerate(refined_fingerprints, start=1)
}

listings_cleaning["duplicate_group_id_refined"] = (
    listings_cleaning["duplicate_fingerprint_refined"]
    .map(refined_group_mapping)
    .astype("Int64")
)

print(
    "Candidatos después del refinamiento:",
    listings_cleaning[
        "flag_duplicate_refined_candidate"
    ].sum(),
)

print(
    "Grupos después del refinamiento:",
    listings_cleaning.loc[
        listings_cleaning[
            "flag_duplicate_refined_candidate"
        ],
        "duplicate_group_id_refined",
    ].nunique(),
)

display(
    listings_cleaning.loc[
        listings_cleaning[
            "flag_duplicate_refined_candidate"
        ],
        "duplicate_group_size_refined",
    ]
    .value_counts()
    .sort_index()
)

Candidatos después del refinamiento: 191
Grupos después del refinamiento: 92


duplicate_group_size_refined
2    170
3     21
Name: count, dtype: int64

### Detectar promociones de obra nueva

In [161]:
# ============================================================
# Identificación de promociones ambiguas
# ============================================================

promotion_pattern = (
    r"\bobra nueva\b|"
    r"\bpromocion(?:es)?\b|"
    r"\bcooperativa\b|"
    r"\bvppl\b|"
    r"\bvppb\b|"
    r"\bviviendas? protegidas?\b|"
    r"\binicio de obra\b|"
    r"\bfin de obra\b|"
    r"\bentrega prevista\b|"
    r"\bproyecto(?:s)? de vivienda\b|"
    r"\bcomercializad[ao]s?\b|"
    r"\bnueva construccion\b|"
    r"\bsobre plano\b"
)

listings_cleaning["flag_promotion_or_new_build"] = (
    listings_cleaning["combined_text_norm_all"]
    .str.contains(
        promotion_pattern,
        regex=True,
        na=False,
    )
)

promotion_group_ids = set(
    listings_cleaning.loc[
        listings_cleaning[
            "flag_duplicate_refined_candidate"
        ]
        & listings_cleaning[
            "flag_promotion_or_new_build"
        ],
        "duplicate_group_id_refined",
    ]
    .dropna()
    .astype(int)
)

listings_cleaning[
    "flag_duplicate_promotion_ambiguous"
] = (
    listings_cleaning[
        "flag_duplicate_refined_candidate"
    ]
    & listings_cleaning[
        "duplicate_group_id_refined"
    ].isin(promotion_group_ids)
)

print(
    "Anuncios de promociones que se conservarán:",
    listings_cleaning[
        "flag_duplicate_promotion_ambiguous"
    ].sum(),
)

print(
    "Grupos de promociones ambiguas:",
    len(promotion_group_ids),
)

Anuncios de promociones que se conservarán: 4
Grupos de promociones ambiguas: 2


### Resolver los duplicados de alta confianza

In [162]:
# ============================================================
# Resolución conservadora de duplicados
# ============================================================

completeness_columns = [
    "ascensor",
    "localizacion",
    "planta",
    "habitaciones",
    "baños",
    "tags",
    "descripcion",
]

listings_cleaning["dedup_completeness_score"] = (
    listings_cleaning[completeness_columns]
    .notna()
    .sum(axis=1)
)

listings_cleaning["dedup_description_length"] = (
    listings_cleaning["descripcion"]
    .fillna("")
    .astype(str)
    .str.len()
)

listings_cleaning["idealista_listing_id"] = pd.to_numeric(
    listings_cleaning["Enlace"]
    .str.extract(
        r"/inmueble/(\d+)/",
        expand=False,
    ),
    errors="coerce",
)

non_promotion_duplicate_mask = (
    listings_cleaning[
        "flag_duplicate_refined_candidate"
    ]
    & ~listings_cleaning[
        "flag_duplicate_promotion_ambiguous"
    ]
)

duplicate_rows_ordered = (
    listings_cleaning.loc[
        non_promotion_duplicate_mask
    ]
    .sort_values(
        [
            "duplicate_group_id_refined",
            "dedup_completeness_score",
            "dedup_description_length",
            "idealista_listing_id",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ],
    )
)

representative_indices = (
    duplicate_rows_ordered
    .groupby(
        "duplicate_group_id_refined",
        sort=False,
    )
    .head(1)
    .index
)

listings_cleaning["duplicate_resolution_status"] = (
    "not_candidate"
)

listings_cleaning.loc[
    listings_cleaning[
        "flag_duplicate_promotion_ambiguous"
    ],
    "duplicate_resolution_status",
] = "retain_promotion_ambiguous"

listings_cleaning.loc[
    non_promotion_duplicate_mask,
    "duplicate_resolution_status",
] = "exclude_high_confidence_duplicate"

listings_cleaning.loc[
    representative_indices,
    "duplicate_resolution_status",
] = "keep_duplicate_representative"

display(
    listings_cleaning[
        "duplicate_resolution_status"
    ].value_counts()
)

duplicate_resolution_status
not_candidate                        11635
exclude_high_confidence_duplicate       97
keep_duplicate_representative           90
retain_promotion_ambiguous               4
Name: count, dtype: int64

### Crear el indicador definitivo y guardar

In [163]:
# Los registros originales se conservan.
# Esta columna indica cuáles se utilizarán en los cálculos.

listings_cleaning["use_after_deduplication"] = (
    listings_cleaning[
        "duplicate_resolution_status"
    ].ne("exclude_high_confidence_duplicate")
)

listings_cleaning[
    "use_for_neighborhood_price_final"
] = (
    listings_cleaning[
        "use_for_neighborhood_price"
    ]
    & listings_cleaning[
        "use_after_deduplication"
    ]
)

print(
    "Duplicados de alta confianza excluidos:",
    listings_cleaning[
        "duplicate_resolution_status"
    ].eq(
        "exclude_high_confidence_duplicate"
    ).sum(),
)

print(
    "Duplicados excluidos que afectaban al cálculo de precios:",
    (
        listings_cleaning[
            "duplicate_resolution_status"
        ].eq(
            "exclude_high_confidence_duplicate"
        )
        & listings_cleaning[
            "use_for_neighborhood_price"
        ]
    ).sum(),
)

print(
    "Anuncios finales utilizables para precios por barrio:",
    listings_cleaning[
        "use_for_neighborhood_price_final"
    ].sum(),
)

Duplicados de alta confianza excluidos: 97
Duplicados excluidos que afectaban al cálculo de precios: 87
Anuncios finales utilizables para precios por barrio: 11109


In [164]:
refined_duplicate_candidates = listings_cleaning.loc[
    listings_cleaning[
        "flag_duplicate_refined_candidate"
    ],
    [
        "duplicate_group_id_refined",
        "duplicate_group_size_refined",
        "duplicate_resolution_status",
        "flag_promotion_or_new_build",
        "titulo",
        "price_eur",
        "area_m2",
        "rooms_clean",
        "bathrooms_clean",
        "floor_number",
        "elevator_status",
        "location_status",
        "property_type",
        "neighborhood_name_final",
        "descripcion",
        "Enlace",
    ],
].copy()

refined_duplicate_candidates.to_csv(
    INTERIM_DIR
    / "duplicate_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig",
)

listings_cleaning.to_csv(
    INTERIM_DIR
    / "listings_duplicate_resolved.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Detección refinada y resolución guardadas.")

Detección refinada y resolución guardadas.


In [165]:
# ============================================================
# Selección definitiva para el análisis inmobiliario por barrio
# ============================================================

listings_cleaning["use_in_neighborhood_model"] = (
    listings_cleaning["use_for_neighborhood_price_final"]
    & listings_cleaning["zone_key_final"].notna()
    & listings_cleaning["price_eur"].notna()
    & listings_cleaning["area_m2"].notna()
    & listings_cleaning["price_per_m2"].notna()
    & ~listings_cleaning[
        "flag_non_residential_property_type"
    ]
)

print("Total de anuncios conservados:", len(listings_cleaning))

print(
    "Anuncios utilizables en el modelo por barrio:",
    listings_cleaning["use_in_neighborhood_model"].sum(),
)

print(
    "Anuncios conservados pero no utilizados:",
    (~listings_cleaning["use_in_neighborhood_model"]).sum(),
)

print(
    "Anuncios sin barrio oficial:",
    listings_cleaning["zone_key_final"].isna().sum(),
)

Total de anuncios conservados: 11826
Anuncios utilizables en el modelo por barrio: 11109
Anuncios conservados pero no utilizados: 717
Anuncios sin barrio oficial: 0


### Validar que la selección es coherente

In [166]:
# ============================================================
# Validación de integridad del dataset limpio
# ============================================================

market_model = listings_cleaning.loc[
    listings_cleaning["use_in_neighborhood_model"]
].copy()

quality_checks = {
    "total_original_conservado": (
        len(listings_cleaning) == 11826
    ),
    "enlaces_originales_unicos": (
        listings_cleaning["Enlace"].duplicated().sum() == 0
    ),
    "modelo_sin_barrio_ausente": (
        market_model["zone_key_final"].isna().sum() == 0
    ),
    "modelo_sin_precio_ausente": (
        market_model["price_eur"].isna().sum() == 0
    ),
    "modelo_sin_superficie_ausente": (
        market_model["area_m2"].isna().sum() == 0
    ),
    "modelo_sin_precio_m2_ausente": (
        market_model["price_per_m2"].isna().sum() == 0
    ),
        "modelo_sin_operaciones_especiales": (
        market_model[
            "flag_special_transaction_all"
        ].sum() == 0
    ),

    "modelo_sin_usos_no_residenciales": (
        market_model[
            "flag_non_residential_use_confirmed"
        ].sum() == 0
    ),

    "modelo_sin_errores_superficie": (
        market_model[
            "flag_probable_area_error_reviewed"
        ].sum() == 0
    ),
    "modelo_sin_duplicados_excluidos": (
        market_model[
            "duplicate_resolution_status"
        ]
        .eq("exclude_high_confidence_duplicate")
        .sum() == 0
    ),
}

quality_checks_df = pd.DataFrame(
    {
        "comprobacion": quality_checks.keys(),
        "resultado": quality_checks.values(),
    }
)

display(quality_checks_df)

if not quality_checks_df["resultado"].all():
    raise ValueError(
        "Alguna comprobación de calidad no se ha superado."
    )

print("Todas las comprobaciones de calidad se han superado.")

,comprobacion,resultado
0,total_original_conservado,True
1,enlaces_originales_unicos,True
2,modelo_sin_barrio_ausente,True
3,modelo_sin_precio_ausente,True
4,modelo_sin_superficie_ausente,True
5,modelo_sin_precio_m2_ausente,True
6,modelo_sin_operaciones_especiales,True
7,modelo_sin_usos_no_residenciales,True
8,modelo_sin_errores_superficie,True
9,modelo_sin_duplicados_excluidos,True


Todas las comprobaciones de calidad se han superado.


### Crear el dataset limpio y trazable

In [167]:
# ============================================================
# Selección de columnas del dataset limpio
# ============================================================

clean_columns = [
    # Identificación y fuente original
    "provincia",
    "zona",
    "titulo",
    "tags",
    "descripcion",
    "Enlace",

    # Valores originales
    "PrecioActual",
    "PrecioAnterior",
    "metros",
    "habitaciones",
    "baños",
    "ascensor",
    "localizacion",
    "planta",

    # Territorialización
    "subzona_portal",
    "zone_key_final",
    "neighborhood_name_final",
    "district_name_final",
    "mapping_method_final",
    "mapping_detail_final",
    "mapping_confidence_final",
    "mapping_status",
    "validation_method",
    "confidence",
    "source_name",
    "source_reference",
    "validation_notes",

    # Precio y superficie
    "price_eur",
    "previous_price_eur",
    "previous_price_available",
    "area_m2",
    "price_per_m2",
    "price_area_action",
    "price_area_action_notes",
    "flag_probable_area_error_reviewed",
    "flag_non_standard_use_reviewed",

    # Habitaciones y baños
    "rooms_clean",
    "rooms_source",
    "bathrooms_clean",
    "bathrooms_source",
    "flag_bathrooms_recovered_text",
    "flag_bathrooms_ambiguous",
    "flag_bathrooms_high_review",

    # Ascensor e interior/exterior
    "elevator_status",
    "elevator_source",
    "elevator_binary",
    "location_status",
    "location_source",
    "exterior_binary",

    # Planta y tipo
    "floor_number",
    "floor_category",
    "floor_source",
    "property_type",
    "property_type_source",
    "flag_non_residential_property_type",

    # Operaciones especiales
    "flag_special_occupied_or_no_possession",
    "flag_special_bare_ownership_or_usufruct",
    "flag_special_partial_ownership",
    "flag_special_auction",
    "flag_special_tenant_or_rented",
    "flag_special_transaction_all",
    "special_transaction_reasons",

    # Posible uso no residencial
    "flag_possible_non_residential_use",
    "flag_non_residential_use_high_confidence",
    "flag_non_residential_use_confirmed",
    "non_residential_use_reasons",

    # Duplicados
    "duplicate_group_id_refined",
    "duplicate_group_size_refined",
    "flag_promotion_or_new_build",
    "flag_duplicate_promotion_ambiguous",
    "duplicate_resolution_status",
    "use_after_deduplication",

    # Decisión final
    "standard_market_status",
    "use_for_neighborhood_price_final",
    "use_in_neighborhood_model",
]

missing_columns = [
    column
    for column in clean_columns
    if column not in listings_cleaning.columns
]

if missing_columns:
    raise ValueError(
        "Faltan columnas necesarias:\n"
        f"{missing_columns}"
    )

listings_clean = listings_cleaning[
    clean_columns
].copy()

listings_market_eligible = listings_clean.loc[
    listings_clean["use_in_neighborhood_model"]
].copy()

print(
    "Filas de listings_clean:",
    len(listings_clean),
)

print(
    "Columnas de listings_clean:",
    len(listings_clean.columns),
)

print(
    "Filas de listings_market_eligible:",
    len(listings_market_eligible),
)

Filas de listings_clean: 11826
Columnas de listings_clean: 75
Filas de listings_market_eligible: 11109


### Guardar los dos datasets

In [168]:
listings_clean_path = (
    PROCESSED_DIR
    / "listings_clean.csv"
)

listings_market_path = (
    PROCESSED_DIR
    / "listings_market_eligible.csv"
)

listings_clean.to_csv(
    listings_clean_path,
    index=False,
    encoding="utf-8-sig",
)

listings_market_eligible.to_csv(
    listings_market_path,
    index=False,
    encoding="utf-8-sig",
)

print("Dataset limpio completo:")
print(listings_clean_path)

print("\nDataset analítico:")
print(listings_market_path)

Dataset limpio completo:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\listings_clean.csv

Dataset analítico:
C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\listings_market_eligible.csv
